<div align="center">

#  Predicción de brotes de dengue en Argentina
## Modelo de aprendizaje automático con variables climáticas y epidemiológicas

---

| | |
|---|---|
| **Integrantes** | *Balda Javier · Caracoix Juan · Casas Facundo* |
| **Institución** | *Universidad Católica Argentina* |
| **Materia / curso** | *Consultoría de datos: Laboratorio III* |
| **Fecha** | Abril 2026 |
| **Datos** | SNVS 2018–2025 (Ministerio de Salud, Argentina) + NASA POWER API |
| **Revista objetivo** | *Revista de Salud UIS* (Universidad Industrial de Santander, Colombia) |

</div>

##  Tabla de contenidos

1. [Objetivo del proyecto](#objetivo)
2. [Hipótesis y preguntas de investigación](#hipotesis)
3. [Estado del arte](#estado-del-arte)
4. [Revista objetivo](#revista)
5. [Configuración del entorno](#config)
6. [Ingesta de datos — SNVS](#snvs)
7. [Ingesta de datos — NASA POWER API](#nasa)
8. [Calidad y limpieza de datos](#limpieza)
9. [Dataset limpio — exportación](#exportacion)
10. [Análisis exploratorio (EDA)](#eda)
11. [Feature engineering](#features)
12. [Modelo baseline y modelo candidato](#modelos)
13. [Validación walk-forward](#validacion)
14. [Resultados e interpretabilidad](#resultados)

---
<a id="objetivo"></a>

##  1. Objetivo del proyecto

Este proyecto tiene como objetivo desarrollar un **modelo predictivo de brotes de dengue a nivel departamental en Argentina**, capaz de anticipar el número de casos confirmados con **2 a 4 semanas de anticipación**, utilizando exclusivamente datos de acceso público.

### Objetivo general

Construir y evaluar un sistema de alerta temprana basado en aprendizaje automático que integre series epidemiológicas semanales del SNVS (2018–2025), variables climáticas de NASA POWER y factores sociodemográficos del Censo 2022, para predecir la incidencia departamental de dengue en Argentina.

### Objetivos específicos

1. Integrar y normalizar datos heterogéneos provenientes de múltiples fuentes públicas (SNVS, NASA POWER, INDEC).
2. Construir un conjunto de features que capture rezagos epidemiológicos, climáticos y contexto socioeconómico.
3. Comparar el desempeño de modelos de referencia (media móvil, Random Forest, XGBoost) mediante validación walk-forward.
4. Evaluar el sistema sobre el brote 2024 —el más severo de la historia argentina— como test prospectivo.
5. Producir un mapa de alertas departamental actualizable semanalmente.

---
<a id="hipotesis"></a>

##  2. Hipótesis y preguntas de investigación

### Pregunta experimental principal

> ¿Es posible predecir el número de casos semanales de dengue por departamento con 2 a 4 semanas de anticipación, alcanzando un **MAPE ≤ 30 %** en departamentos con más de 100 casos históricos acumulados, usando variables climáticas rezagadas, índice de vegetación NDVI y casos previos, superando un baseline de media móvil histórica?

### Hipótesis principal

**H1:** Un modelo de ensamble (Random Forest / XGBoost) entrenado con lags epidemiológicos y variables climáticas rezagadas (t−2 a t−6 semanas) alcanza un MAPE significativamente menor que una media móvil de 4 semanas en departamentos con más de 100 casos históricos.

### Hipótesis secundarias

**H2:** La temperatura media rezagada 2–4 semanas es el predictor climático de mayor importancia, consistente con el ciclo de vida del *Aedes aegypti* (~14 días entre picadura y viremia).

**H3:** El índice NBI (Necesidades Básicas Insatisfechas) y la densidad poblacional mejoran la predicción al controlar heterogeneidad socioeconómica entre departamentos.

**H4:** El modelo entrenado en 2018–2023 generaliza al brote 2024 mejor que el baseline, demostrando capacidad predictiva en condiciones de brote sin precedentes.

### Preguntas adicionales de análisis

- ¿Qué lag temporal maximiza la correlación entre temperatura/precipitación y casos?
- ¿Existen diferencias regionales en la importancia de las variables climáticas (NOA vs. región pampeana vs. NEA)?
- ¿El modelo detecta el inicio del brote (alerta temprana binaria) con precisión suficiente para guiar operaciones de fumigación?

---
<a id="estado-del-arte"></a>

##  3. Estado del arte y marco conceptual

### 3.1 Epidemiología del dengue y contexto global

El dengue es la arbovirosis de mayor expansión a nivel mundial, con aproximadamente 390 millones de infecciones anuales en más de 100 países (Bhatt et al., 2013). La Organización Mundial de la Salud reporta que la incidencia global se multiplicó por diez entre 2000 y 2019. En América Latina, Brasil, Colombia, Argentina y México concentran la mayor carga de enfermedad. Argentina registró en 2024 su brote más severo, con 582.267 casos confirmados, lo que pone en evidencia la expansión del vector hacia latitudes históricamente no endémicas.

### 3.2 Modelos estadísticos clásicos

Los primeros enfoques predictivos se basaron en modelos de series temporales univariados. Luz et al. (2008) analizaron la incidencia de dengue en Río de Janeiro mediante modelos ARIMA, estableciendo una de las primeras referencias metodológicas para la región. Los modelos SARIMA y SARIMAX incorporan posteriormente la estacionalidad y variables climáticas exógenas (temperatura, humedad), mejorando la captura de patrones cíclicos. Racloz et al. (2012) revisaron sistemáticamente los modelos de vigilancia epidemiológica y señalaron que los enfoques univariados sub-representan la dinámica espacial y las interacciones entre variables.

### 3.3 Aprendizaje automático aplicado a la predicción de dengue

La incorporación de técnicas de machine learning marcó un punto de inflexión. Mussumeci y Coelho (2020) compararon modelos LSTM, Random Forest y regresión Lasso para predicción a gran escala en Brasil, encontrando que Random Forest ofrecía el mejor balance entre precisión y interpretabilidad para horizontes cortos (1–4 semanas). Roster et al. (2022) validaron modelos de machine learning en ciudades brasileñas usando variables epidemiológicas y meteorológicas, destacando que los modelos de ensamble superan a la regresión de Poisson en predicción de corto plazo. Salim et al. (2021) evaluaron SVM, árbol de decisión y redes neuronales para Selangor (Malasia), alcanzando 70% de precisión con SVM; sin embargo, los modelos de ensamble como XGBoost y Random Forest superaron a los clasificadores individuales (Patil y Pandya, 2021; Mazhar et al., 2024).

Sebastianelli et al. (2024) propusieron un enfoque de ensamble reproducible para Brasil, demostrando transferibilidad a Perú y estableciendo un marco metodológico aplicable a Argentina. Más recientemente, Jiménez et al. (2025) desarrollaron un modelo LSTM con SHAP para Brasil que integra lags climáticos y efectos espaciales entre estados vecinos, logrando predicciones a 4 y 12 semanas con ventanas deslizantes de 7 años de entrenamiento.

### 3.4 Variables climáticas y rezagos temporales

La temperatura es el predictor climático dominante en la biología del *Aedes aegypti*: el ciclo completo de vida (huevo → adulto) tarda aproximadamente 7–14 días a 25–30°C, mientras que el período de incubación extrínseca del virus se acorta drásticamente por encima de 30°C. Esto fundamenta el uso de rezagos climáticos de 2–6 semanas como features (Cabrera et al., 2022; Salim et al., 2021). La precipitación acumulada favorece la formación de criaderos, aunque con efectos no lineales: precipitaciones muy intensas pueden eliminar larvas por arrastre. Ochida et al. (2022) identificaron que la humedad relativa mínima y la temperatura máxima son los predictores climáticos de mayor importancia en contextos tropicales.

### 3.5 Revisiones sistemáticas recientes

Leung et al. (2023) realizaron la revisión sistemática más completa hasta la fecha sobre modelos de predicción de brotes de dengue, analizando 160 estudios. Concluyeron que los modelos de aprendizaje automático superan consistentemente a los métodos estadísticos clásicos, y que la incorporación de variables ambientales y satelitales (NDVI) mejora la precisión, especialmente para horizontes de 2–4 semanas. Sylvestre et al. (2022) identificaron en otra revisión sistemática que los datos de vigilancia epidemiológica combinados con variables meteorológicas constituyen la combinación de inputs más robusta. Mazhar et al. (2024) destacaron que los modelos basados en datos reales de vigilancia (como el SNVS de Argentina) superan a los modelos teóricos compartimentales (SIR/SEIR) en precisión operativa.

### 3.6 Vacío en la literatura y aporte de este trabajo

La mayoría de los estudios existentes se focalizan en Brasil, Colombia o países del sudeste asiático. **No se identificaron publicaciones que apliquen modelos de machine learning con validación walk-forward al sistema de vigilancia argentino (SNVS) utilizando datos departamentales reales**, ni que evalúen el brote 2024 como test prospectivo. Este trabajo busca cubrir ese vacío, constituyendo el primer análisis sistemático de predicción departamental de dengue en Argentina con datos reales 2018–2025.

---

### Referencias (formato APA 7.ª edición)

Bhatt, S., Gething, P. W., Brady, O. J., Messina, J. P., Farlow, A. W., Moyes, C. L., Drake, J. M., Brownstein, J. S., Hoen, A. G., Sankoh, O., Myers, M. F., George, D. B., Jaenisch, T., Wint, G. R. W., Simmons, C. P., Scott, T. W., Farrar, J. J., & Hay, S. I. (2013). The global distribution and burden of dengue. *Nature*, *496*(7446), 504–507. https://doi.org/10.1038/nature12060

Cabrera, M., Leake, J., Naranjo-Torres, J., Valero, N., Cabrera, J. C., & Rodríguez-Morales, A. J. (2022). When climate variables improve the dengue forecasting: A machine learning approach. *The European Physical Journal Special Topics*, *231*, 3473–3485. https://doi.org/10.1140/epjs/s11734-024-01201-7

Jiménez, L., Costa, G. P., & Silva, F. R. (2025). Forecasting dengue across Brazil with LSTM neural networks and SHAP-driven lagged climate and spatial effects. *BMC Public Health*, *25*, 892. https://doi.org/10.1186/s12889-025-22106-7

Leung, X. Y., Islam, R. M., Adhami, M., Ilic, D., McDonald, L., Palawaththa, S., Diug, B., Munshi, S. U., & Karim, M. N. (2023). A systematic review of dengue outbreak prediction models: Current scenario and future directions. *PLOS Neglected Tropical Diseases*, *17*(2), e0010631. https://doi.org/10.1371/journal.pntd.0010631

Luz, P. M., Mendes, B. V. M., Codeço, C. T., Struchiner, C. J., & Galvani, A. P. (2008). Time series analysis of dengue incidence in Rio de Janeiro, Brazil. *The American Journal of Tropical Medicine and Hygiene*, *79*(6), 933–939. https://doi.org/10.4269/ajtmh.2008.79.933

Mazhar, B., Mazhar Ali, N., Manzoor, F., Khan, M. K., Nasir, M., & Ramzan, M. (2024). Development of data-driven machine learning models and their potential role in predicting dengue outbreak. *Journal of Vector Borne Diseases*, *61*(4), 503–514. https://doi.org/10.4103/0972-9062.393976

Mussumeci, E., & Coelho, F. C. (2020). Large-scale multivariate forecasting models for dengue – LSTM versus random forest regression. *Spatial and Spatio-temporal Epidemiology*, *35*, 100372. https://doi.org/10.1016/j.sste.2020.100372

Ochida, N., Mangeas, M., Dupont-Rouzeyrol, M., Dutheil, C., Forfait, C., Peltier, A., Descloux, E., & Menkes, C. (2022). Identification of significant climatic risk factors and machine learning models in dengue outbreak prediction. *BMC Medical Informatics and Decision Making*, *21*, 141. https://doi.org/10.1186/s12911-021-01493-y

Patil, S., & Pandya, S. (2021). Forecasting dengue hotspots associated with variation in meteorological parameters using regression and time series models. *Frontiers in Public Health*, *9*, 798034. https://doi.org/10.3389/fpubh.2021.798034

Racloz, V., Ramsey, R., Tong, S., & Hu, W. (2012). Surveillance of dengue fever virus: A review of epidemiological models and early warning systems. *PLOS Neglected Tropical Diseases*, *6*(5), e1648. https://doi.org/10.1371/journal.pntd.0001648

Roster, K., Connaughton, C., & Rodrigues, F. A. (2022). Machine-learning–based forecasting of dengue fever in Brazilian cities using epidemiologic and meteorological variables. *American Journal of Epidemiology*, *191*(10), 1803–1812. https://doi.org/10.1093/aje/kwac090

Salim, N. A. M., Wah, Y. B., Reeves, C., Smith, M., Yaacob, W. F. W., Mudin, R. N., Dapari, R., Sapri, N. N. F. F., & Haque, U. (2021). Prediction of dengue outbreak in Selangor Malaysia using machine learning techniques. *Scientific Reports*, *11*, 939. https://doi.org/10.1038/s41598-020-79193-2

Sebastianelli, A., Spiller, D., Carmo, R., Wheeler, J., Nowakowski, A., Jacobson, L. V., Kim, D., Barlevi, H., Cordero, Z. E., Colón-González, F. J., & Lowe, R. (2024). A reproducible ensemble machine learning approach to forecast dengue outbreaks. *Scientific Reports*, *14*, 3807. https://doi.org/10.1038/s41598-024-52796-9

Sylvestre, E., Joachim, C., Cécilia-Joseph, E., Bouzillé, G., Campillo-Gimenez, B., Cuggia, M., & Cabié, A. (2022). Data-driven methods for dengue prediction and surveillance using real-world and Big Data: A systematic review. *PLOS Neglected Tropical Diseases*, *16*(1), e0010056. https://doi.org/10.1371/journal.pntd.0010056

---
<a id="revista"></a>

##  4. Revista objetivo

**Revista de Salud UIS** — Universidad Industrial de Santander, Colombia  
🔗 https://revistas.uis.edu.co/index.php/revistasaluduis

| Campo | Detalle |
|---|---|
| **ISSN impreso** | 0121-0807 |
| **ISSN electrónico** | 2145-8464 |
| **Indexación** | SciELO Colombia, Redalyc, LILACS, Latindex Catálogo |
| **Idioma** | Español (resumen en inglés) |
| **Periodicidad** | Trimestral |
| **Alcance** | Salud pública, epidemiología, medicina preventiva, ciencias de la salud |
| **Formato de citación** | Vancouver / APA |

### Justificación de la elección

La *Revista de Salud UIS* es un espacio de publicación reconocido en América Latina para estudios de epidemiología aplicada y salud pública. Su alcance temático coincide directamente con el contenido de este trabajo: modelado predictivo de enfermedades transmisibles con impacto en salud pública regional. La indexación en SciELO y LILACS garantiza visibilidad en la región donde se producen los brotes estudiados. Además, el enfoque metodológico cuantitativo con datos reales de vigilancia sanitaria se alinea con el perfil de trabajos publicados recientemente en la revista.

---
<a id="config"></a>

##  5. Configuración del entorno

In [20]:
# Instalar dependencias no incluidas en Colab por defecto
!pip install -q xgboost

---
<a id="snvs"></a>

##  6. Ingesta de datos — SNVS

El **Sistema Nacional de Vigilancia de la Salud (SNVS 2.0)** publica casos confirmados de dengue por departamento y semana epidemiológica desde 2018.  
URL: https://snvs2.msal.gob.ar

Los archivos presentan **dos esquemas de columnas** según el año:

| Esquema | Años | Columnas clave |
|---|---|---|
| **A (original)** | 2018–2022 | `departamento_nombre`, `semanas_epidemiologicas`, `cantidad_casos` |
| **B (nuevo)** | 2023–2025 | `departamento_residencia`, `sepi_min`, `cantidad` |

La función `load_snvs_file()` detecta el esquema automáticamente.

In [24]:
# ── Mapas de normalización de columnas ────────────────────────────────────────
SCHEMA_A = {
    'departamento_id':         'depto_id',
    'departamento_nombre':     'departamento',
    'provincia_id':            'prov_id',
    'provincia_nombre':        'provincia',
    'año':                     'anio',
    'ano':                     'anio',
    'semanas_epidemiologicas': 'semana',
    'evento_nombre':           'evento',
    'grupo_edad_id':           'grupo_edad_id',
    'grupo_edad_desc':         'grupo_edad_desc',
    'cantidad_casos':          'cantidad_casos',
}
SCHEMA_B = {
    'id_depto_indec_residencia':  'depto_id',
    'departamento_residencia':    'departamento',
    'id_prov_indec_residencia':   'prov_id',
    'provincia_residencia':       'provincia',
    'anio_min':                   'anio',
    'evento':                     'evento',
    'id_grupo_etario':            'grupo_edad_id',
    'grupo_etario':               'grupo_edad_desc',
    'sepi_min':                   'semana',
    'cantidad':                   'cantidad_casos',
}

def _clean_col(c: str) -> str:
    """Elimina BOM, comillas y espacios de nombres de columna."""
    return c.strip().lstrip('\ufeff').strip('"').lower().replace('ï»¿', '')


def load_snvs_file(path: Path) -> pd.DataFrame:
    """
    Carga un archivo SNVS (CSV o XLSX), normaliza columnas,
    filtra solo casos de Dengue y agrega por departamento × semana.

    Retorna DataFrame con columnas canónicas:
    anio | semana | depto_id | departamento | prov_id | provincia | cantidad_casos
    """
    suffix = path.suffix.lower()
    if suffix == '.csv':
        df = pd.read_csv(path, encoding='latin-1', sep=None, engine='python')
    elif suffix in ('.xlsx', '.xls'):
        df = pd.read_excel(path)
    else:
        raise ValueError(f'Formato no soportado: {suffix}')

    df.columns = [_clean_col(c) for c in df.columns]

    rename = {}
    for col in df.columns:
        if col in SCHEMA_A:   rename[col] = SCHEMA_A[col]
        elif col in SCHEMA_B: rename[col] = SCHEMA_B[col]
    df = df.rename(columns=rename)

    # Filtrar solo dengue
    if 'evento' in df.columns:
        df = df[df['evento'].astype(str).str.lower().str.strip() == 'dengue'].copy()

    # Agregar por departamento × semana (sumar grupos de edad)
    key_cols = [c for c in ['anio', 'semana', 'depto_id', 'departamento',
                             'prov_id', 'provincia'] if c in df.columns]
    df = df.groupby(key_cols, as_index=False)['cantidad_casos'].sum()

    df['anio']          = df['anio'].astype(float).astype(int)
    df['semana']        = df['semana'].astype(float).astype(int)
    df['cantidad_casos']= df['cantidad_casos'].astype(float).astype(int)
    return df


print('Funciones de normalización SNVS definidas.')

Funciones de normalización SNVS definidas.


In [ ]:
# Configuración de rutas local en Codespaces
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == 'notebooks' and (BASE_DIR.parent / 'data').exists():
    BASE_DIR = BASE_DIR.parent
DATA_DIR = BASE_DIR / 'data' / 'dengue'
OUT_DIR = BASE_DIR / 'data'
print(f'Directorio de datos: {DATA_DIR}')
print(f'Directorio de salida: {OUT_DIR}')


In [25]:
ARCHIVOS_SNVS = [
    'dengue-zika-nacional-2018.csv',
    'dengue-zika-nacional-2019.xlsx',
    'dengue-zika-nacional-2020.xlsx',
    'dengue-zika-nacional-2021.xlsx',
    'dengue-zika-nacional-2022.csv',
    'dengue-zika-nacional-2023.csv',
    'dengue-zika-nacional-2024.csv',
    'dengue-zika-nacional-2025-primer-semestre.csv',
    'dengue-zika-nacional-2025-segundo-semestre.csv',
    # Nota: 2024-(1).csv y 2024-(2).csv son versiones alternativas
    # del mismo período. NO incluir junto con 2024.csv.
]

dfs = []
for fname in ARCHIVOS_SNVS:
    path = DATA_DIR / fname
    if not path.exists():
        print(f'  ⚠️  Omitido (no encontrado): {fname}')
        continue
    try:
        d = load_snvs_file(path)
        dfs.append(d)
        print(f'  ✅ {fname}: {len(d):,} filas | {d["cantidad_casos"].sum():,} casos')
    except Exception as e:
        print(f'  ❌ {fname}: {e}')

df_raw = pd.concat(dfs, ignore_index=True)

print(f'\n📊 Dataset SNVS cargado: {len(df_raw):,} filas | {df_raw.cantidad_casos.sum():,} casos totales')

  ✅ dengue-zika-nacional-2018.csv: 1,096 filas | 17,576 casos
  ✅ dengue-zika-nacional-2019.xlsx: 2,732 filas | 46,437 casos
  ❌ dengue-zika-nacional-2020.xlsx: Excel file format cannot be determined, you must specify an engine manually.
  ✅ dengue-zika-nacional-2021.xlsx: 271 filas | 1,164 casos
  ✅ dengue-zika-nacional-2022.csv: 384 filas | 1,576 casos
  ✅ dengue-zika-nacional-2023.csv: 7,245 filas | 582,265 casos
  ✅ dengue-zika-nacional-2024.csv: 7,229 filas | 581,433 casos
  ⚠️  Omitido (no encontrado): dengue-zika-nacional-2025-primer-semestre.csv
  ❌ dengue-zika-nacional-2025-segundo-semestre.csv: Expected 2 fields in line 7, saw 3

📊 Dataset SNVS cargado: 18,957 filas | 1,230,451 casos totales


---
<a id="nasa"></a>

##  7. Ingesta de datos — NASA POWER API

**NASA POWER** provee datos climáticos diarios para cualquier coordenada sin API key ni registro.  
Documentación: https://power.larc.nasa.gov/api/temporal/daily/point

Se descargan **20 parámetros** (límite máximo por request) seleccionados por su relevancia en la biología del *Aedes aegypti*.

In [26]:
# ── Parámetros seleccionados (mínimo esencial para evitar errores 422) ─────────────
NASA_PARAMS = ','.join([
    # Temperatura (1)
    'T2M',
    # Agua y humedad (2)
    'PRECTOTCORR', 'RH2M',
    # Fecha (2) - para robustez en construccion de fecha
    'MO', 'DY'
])

# Centroides por provincia (lat, lon)
CENTROIDES_PROV = {
    'Buenos Aires':        (-36.60, -60.00),
    'CABA':                (-34.61, -58.44),
    'Córdoba':             (-31.42, -64.18),
    'Santa Fe':            (-31.63, -60.70),
    'Tucumán':             (-26.82, -65.22),
    'Salta':               (-24.79, -65.41),
    'Misiones':            (-27.36, -55.90),
    'Jujuy':               (-24.18, -65.30),
    'Entre Ríos':          (-31.74, -60.54),
    'Chaco':               (-27.46, -58.99),
    'Formosa':             (-26.18, -58.18),
    'Corrientes':          (-27.47, -58.83),
    'Santiago del Estero': (-27.78, -64.26),
}

print(f'Parámetros a descargar: {len(NASA_PARAMS.split(","))} variables')
print(f'Provincias: {len(CENTROIDES_PROV)}')


Parámetros a descargar: 5 variables
Provincias: 13


In [27]:
# ── 20 parámetros seleccionados (límite máximo de la API) ─────────────────────
NASA_PARAMS = ','.join([
    # Temperatura (6)
    'T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'T_WET_RANGE_2M', 'TS',
    # Agua y humedad (6)
    'PRECTOTCORR', 'PRECTOTCORR_SUM', 'RH2M', 'RH2M_MAX', 'RH2M_MIN', 'QV2M',
    # Radiación y viento (4)
    'ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_LW_DWN', 'WS2M', 'WS2M_MAX',
    # Auxiliares (4)
    'PS', 'GWETROOT', 'EVPTRNS', 'T2MDEW',
])

# Centroides por provincia (lat, lon)
CENTROIDES_PROV = {
    'Buenos Aires':        (-36.60, -60.00),
    'CABA':                (-34.61, -58.44),
    'Córdoba':             (-31.42, -64.18),
    'Santa Fe':            (-31.63, -60.70),
    'Tucumán':             (-26.82, -65.22),
    'Salta':               (-24.79, -65.41),
    'Misiones':            (-27.36, -55.90),
    'Jujuy':               (-24.18, -65.30),
    'Entre Ríos':          (-31.74, -60.54),
    'Chaco':               (-27.46, -58.99),
    'Formosa':             (-26.18, -58.18),
    'Corrientes':          (-27.47, -58.83),
    'Santiago del Estero': (-27.78, -64.26),
}

print(f'Parámetros a descargar: {len(NASA_PARAMS.split(","))} variables')
print(f'Provincias: {len(CENTROIDES_PROV)}')

Parámetros a descargar: 20 variables
Provincias: 13


In [28]:
NASA_PARAMS = ','.join([
    'T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE',
    'TS',
    'PRECTOTCORR',
    'RH2M', 'QV2M',                          # ← eliminados RH2M_MAX y RH2M_MIN
    'ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_LW_DWN',
    'WS2M', 'WS2M_MAX',
    'PS', 'GWETROOT', 'EVPTRNS', 'T2MDEW'
])


def fetch_nasa_power(lat: float, lon: float,
                     start_date: str = '20180101',
                     end_date:   str = '20241231',
                     retries: int = 3) -> pd.DataFrame:
    """
    Descarga datos climáticos DIARIOS de NASA POWER para una coordenada.
    Formato CSV — evita pivoteo de JSON. Sin API key.

    Parámetros eliminados por no existir en endpoint daily:
    - RH2M_MAX, RH2M_MIN → no disponibles en community=AG daily
    """
    url = (
        f'https://power.larc.nasa.gov/api/temporal/daily/point'
        f'?parameters={NASA_PARAMS}'
        f'&community=AG'
        f'&longitude={lon}'
        f'&latitude={lat}'
        f'&start={start_date}'
        f'&end={end_date}'
        f'&format=CSV'
    )

    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=120)
            resp.raise_for_status()
            break
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f'    Reintento {attempt + 1}/{retries}... ({e})')
            time.sleep(5)

    lines = resp.text.splitlines()
    try:
        header_idx = next(i for i, line in enumerate(lines) if line.startswith('YEAR'))
    except StopIteration:
        raise ValueError('Respuesta inesperada de NASA POWER — no se encontró encabezado CSV')

    df = pd.read_csv(
        StringIO('\n'.join(lines[header_idx:])),
        na_values='-999',
    )

    df['fecha'] = pd.to_datetime(
        df['YEAR'].astype(str) + df['DOY'].astype(str).str.zfill(3),
        format='%Y%j'
    )
    df['anio']   = df['fecha'].dt.isocalendar().year.astype(int)
    df['semana'] = df['fecha'].dt.isocalendar().week.astype(int)

    df = df.rename(columns={
        'T2M':              'temp_media',
        'T2M_MAX':          'temp_max',
        'T2M_MIN':          'temp_min',
        'T2M_RANGE':        'temp_rango',
        'TS':               'temp_superficie',
        'PRECTOTCORR':      'precipitacion_mm',
        'RH2M':             'humedad_media',
        'QV2M':             'humedad_especifica',
        'ALLSKY_SFC_SW_DWN':'radiacion_solar',
        'ALLSKY_SFC_LW_DWN':'radiacion_onda_larga',
        'WS2M':             'viento_medio',
        'WS2M_MAX':         'viento_max',
        'PS':               'presion',
        'GWETROOT':         'humedad_suelo',
        'EVPTRNS':          'evapotranspiracion',
        'T2MDEW':           'punto_rocio',
    })

    clima_vars = [
        'temp_media', 'temp_max', 'temp_min', 'temp_rango',
        'temp_superficie', 'precipitacion_mm',
        'humedad_media', 'humedad_especifica',
        'radiacion_solar', 'radiacion_onda_larga',
        'viento_medio', 'viento_max',
        'presion', 'humedad_suelo', 'evapotranspiracion', 'punto_rocio'
    ]
    clima_vars = [v for v in clima_vars if v in df.columns]

    agg_dict = {v: 'sum' if 'precipitacion' in v else 'mean' for v in clima_vars}
    df_sem = df.groupby(['anio', 'semana']).agg(agg_dict).reset_index()
    df_sem['precipitacion_acum'] = df_sem['precipitacion_mm']

    return df_sem

In [29]:
# ── Celda de descarga ────────────────────────────────────────────────────────
clima_cache = OUT_DIR / 'clima_nasa_power_2018_2024.csv'

if not clima_cache.exists():
    clima_dfs = []
    for prov, (lat, lon) in CENTROIDES_PROV.items():
        print(f'  Descargando {prov}...')
        try:
            d = fetch_nasa_power(lat=lat, lon=lon,
                                start_date='20180101', end_date='20241231')
            d['provincia'] = prov
            clima_dfs.append(d)
            print(f'    OK: {len(d):,} semanas')
            time.sleep(1)
        except Exception as e:
            print(f'    ERROR: {e}')

    df_clima = pd.concat(clima_dfs, ignore_index=True)
    df_clima.to_csv(clima_cache, index=False)
    print(f'Guardado: {clima_cache}')
else:
    df_clima = pd.read_csv(clima_cache)
    print(f'Cargado desde caché: {len(df_clima):,} filas')

Cargado desde caché: 4,758 filas


In [42]:
df_modelado = df_raw.merge(
    df_clima,
    on=['anio', 'semana', 'provincia'],
    how='left'
)

print(f'✅ df_modelado: {len(df_modelado):,} filas | {df_modelado.shape[1]} columnas')
print(f'   Sin clima (NaN): {df_modelado["temp_media"].isna().sum():,} filas')
print(df_modelado.head(3))

✅ df_modelado: 18,957 filas | 24 columnas
   Sin clima (NaN): 3,931 filas
   anio  semana depto_id departamento  prov_id provincia  cantidad_casos  \
0  2025       1       14      Capital   14.000   Córdoba               1   
1  2025       1       14      Formosa   34.000   Formosa               6   
2  2025       1       28      Capital   66.000     Salta               1   

   temp_media  temp_max  temp_min  temp_rango  temp_superficie  \
0      24.420    33.265    16.700      16.565           24.375   
1      28.715    36.205    20.855      15.350           29.220   
2      18.605    24.545    12.720      11.825           20.075   

   precipitacion_mm  humedad_media  humedad_especifica  radiacion_solar  \
0            19.340         56.685              10.870           27.905   
1             0.000         47.690              11.005           31.490   
2             0.030         67.385              11.030           25.535   

   radiacion_onda_larga  viento_medio  viento_max  pres

In [43]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['departamento', 'anio', 'semana']).copy()
    g  = df.groupby('departamento')

    # Lags epidemiológicos
    for lag in [1, 2, 3, 4]:
        df[f'casos_lag{lag}'] = g['cantidad_casos'].shift(lag)

    # Media móvil 4 SE (baseline)
    df['casos_ma4'] = g['cantidad_casos'].transform(
        lambda x: x.shift(1).rolling(4, min_periods=1).mean()
    )

    # Clima rezagado
    for var in ['temp_media', 'temp_max', 'humedad_media', 'precipitacion_mm']:
        if var not in df.columns:
            continue
        for lag in [2, 4, 6]:
            df[f'{var}_lag{lag}'] = g[var].shift(lag)

    if 'punto_rocio' in df.columns:
        df['punto_rocio_lag2'] = g['punto_rocio'].shift(2)
        df['punto_rocio_lag4'] = g['punto_rocio'].shift(4)

    if 'humedad_suelo' in df.columns:
        df['humedad_suelo_lag2'] = g['humedad_suelo'].shift(2)

    # Temperatura acumulada (GDD proxy)
    if 'temp_media' in df.columns:
        df['gdd_lag2'] = g['temp_media'].transform(
            lambda x: x.shift(2).rolling(3, min_periods=1).mean()
        )

    # Precipitación acumulada 3 SE
    if 'precipitacion_mm' in df.columns:
        df['precip_acum3_lag2'] = g['precipitacion_mm'].transform(
            lambda x: x.shift(2).rolling(3, min_periods=1).sum()
        )

    # Codificación cíclica de semana
    df['semana_sin'] = np.sin(2 * np.pi * df['semana'] / 52)
    df['semana_cos'] = np.cos(2 * np.pi * df['semana'] / 52)

    # Indicador de verano austral (SE 1–15 y SE 45–52)
    df['es_verano'] = df['semana'].apply(lambda w: 1 if (w <= 15 or w >= 45) else 0)

    # ID numérico de departamento
    df['depto_code'] = df['departamento'].astype('category').cat.codes.astype(int)

    return df

df_feat = build_features(df_modelado)

# Lista de features final (excluye columnas no disponibles)
CANDIDATES = [
    'casos_lag1','casos_lag2','casos_lag3','casos_lag4','casos_ma4',
    'temp_media_lag2','temp_media_lag4','temp_media_lag6',
    'temp_max_lag2','temp_max_lag4',
    'humedad_media_lag2','humedad_media_lag4',
    'precipitacion_mm_lag2','precipitacion_mm_lag4',
    'precip_acum3_lag2','gdd_lag2',
    'punto_rocio_lag2','punto_rocio_lag4',
    'humedad_suelo_lag2',
    'semana_sin','semana_cos','es_verano','depto_code',
]
FEATURE_COLS = [f for f in CANDIDATES if f in df_feat.columns]
TARGET = 'cantidad_casos'

df_model = df_feat[FEATURE_COLS + [TARGET, 'anio', 'semana', 'departamento', 'casos_ma4']].dropna()
print(f'✅ Dataset modelable: {len(df_model):,} filas | {len(FEATURE_COLS)} features')
print(f'   Burn-in eliminado (NaN por lags): {len(df_feat) - len(df_model):,} filas')
print(f'   Features: {FEATURE_COLS}')

✅ Dataset modelable: 12,488 filas | 23 features
   Burn-in eliminado (NaN por lags): 6,469 filas
   Features: ['casos_lag1', 'casos_lag2', 'casos_lag3', 'casos_lag4', 'casos_ma4', 'temp_media_lag2', 'temp_media_lag4', 'temp_media_lag6', 'temp_max_lag2', 'temp_max_lag4', 'humedad_media_lag2', 'humedad_media_lag4', 'precipitacion_mm_lag2', 'precipitacion_mm_lag4', 'precip_acum3_lag2', 'gdd_lag2', 'punto_rocio_lag2', 'punto_rocio_lag4', 'humedad_suelo_lag2', 'semana_sin', 'semana_cos', 'es_verano', 'depto_code']


In [52]:
import unicodedata

def normalizar_provincia(s: str) -> str:
    """Limpia encoding roto y estandariza nombres."""
    if not isinstance(s, str):
        return 'Desconocida'
    # Reparar encoding latin1 mal interpretado como utf-8
    try:
        s = s.encode('latin1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass
    # Normalizar unicode y strip
    s = unicodedata.normalize('NFC', s).strip()
    # Estandarizar variantes
    aliases = {
        'desconocida': 'Desconocida',
        'DESCONOCIDA': 'Desconocida',
        '*sin dato*':  'Desconocida',
        '(en blanco)': 'Desconocida',
    }
    return aliases.get(s, s)

df_raw['provincia']   = df_raw['provincia'].map(normalizar_provincia)
df_clima['provincia'] = df_clima['provincia'].map(normalizar_provincia)

# Verificar que ya no haya duplicados en df_clima
print(f"Duplicados en df_clima: {df_clima.duplicated(subset=['anio','semana','provincia']).sum()}")

# Re-hacer el merge
df_modelado = df_raw.merge(df_clima, on=['anio', 'semana', 'provincia'], how='left')
df_modelado = df_modelado.drop_duplicates(subset=['anio', 'semana', 'departamento'])

print(f'df_modelado: {len(df_modelado):,} filas')
print(f'Sin clima (provincias sin centroide): {df_modelado["temp_media"].isna().sum():,} filas')

# Provincias que quedaron sin datos climáticos
sin_clima = df_modelado[df_modelado['temp_media'].isna()]['provincia'].unique()
print(f'Provincias sin datos climáticos: {sorted(sin_clima)}')

Duplicados en df_clima: 0
df_modelado: 10,175 filas
Sin clima (provincias sin centroide): 2,294 filas
Provincias sin datos climáticos: ['Buenos Aires', 'CABA', 'Catamarca', 'Chaco', 'Chubut', 'Corrientes', 'Córdoba', 'Desconocida', 'Entre Ríos', 'Formosa', 'La Pampa', 'La Rioja', 'Mendoza', 'Misiones', 'Neuquén', 'Río Negro', 'Salta', 'San Juan', 'San Luis', 'Santa Cruz', 'Santa Fe', 'Santiago del Estero', 'Tierra del Fuego', 'Tucumán']


In [70]:
# 1. Limpieza de datos originales
invalid_deptos = ['(en blanco)', '*sin dato*', 'Desconocida', 'desconocida', 'DESCONOCIDA']
df_raw_clean = df_raw[~df_raw['departamento'].isin(invalid_deptos)].copy()

# 2. Integración y eliminación de duplicados
# Clave única real: depto_id + provincia (no solo departamento, que puede repetirse entre provincias)
df_modelado = (
    df_raw_clean
    .merge(df_clima, on=['anio', 'semana', 'provincia'], how='left')
    .drop_duplicates(subset=['anio', 'semana', 'depto_id', 'provincia'])
)

print(f'df_modelado: {len(df_modelado):,} filas')
print(f'Duplicados restantes: {df_modelado.duplicated(subset=["anio","semana","depto_id","provincia"]).sum()}')

# 3. Feature engineering
df_feat = build_features(df_modelado)

# 4. Selección y limpieza
FEATURE_COLS = [f for f in CANDIDATES if f in df_feat.columns]
METADATA_COLS = ['anio', 'semana', 'departamento', 'casos_ma4']
df_model = df_feat[FEATURE_COLS + [TARGET] + METADATA_COLS].dropna()

# 5. Split
train = df_model[df_model['anio'] < 2024]
test  = df_model[df_model['anio'] == 2024]

X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET].values
y_baseline       = test['casos_ma4'].values.ravel()

print(f'\nDataset preparado:')
print(f' -> Entrenamiento: {X_train.shape[0]:,} filas')
print(f' -> Test:          {X_test.shape[0]:,} filas')
print(f' -> Features:      {len(FEATURE_COLS)}')
print(f' -> y_test:        {len(y_test):,}')
print(f' -> y_baseline:    {len(y_baseline):,}')

df_modelado: 11,434 filas
Duplicados restantes: 0

Dataset preparado:
 -> Entrenamiento: 1,685 filas
 -> Test:          4,818 filas
 -> Features:      23
 -> y_test:        4,818
 -> y_baseline:    9,636


In [71]:
dups_raw = df_raw[df_raw.duplicated(subset=['anio','semana','departamento'], keep=False)]
print(f"Filas duplicadas en df_raw: {len(dups_raw)}")
print(dups_raw.sort_values(['departamento','anio','semana']).head(10))

Filas duplicadas en df_raw: 15389
      anio  semana depto_id departamento  prov_id    provincia  cantidad_casos
1384  2020       8        0  (en blanco)    0.000  Desconocida               1
1385  2020       8       30  (en blanco)   30.000   Entre Ríos               1
1386  2020       8       34  (en blanco)   34.000      Formosa               1
1639  2020      10        0  (en blanco)    0.000  Desconocida               1
1642  2020      10       14  (en blanco)   14.000      Córdoba               1
1643  2020      10       66  (en blanco)   66.000        Salta               1
1794  2020      11       14  (en blanco)   14.000      Córdoba               1
1795  2020      11       66  (en blanco)   66.000        Salta               2
1796  2020      11     2001  (en blanco)    2.000         CABA               1
1976  2020      12       34  (en blanco)   34.000      Formosa               2


In [72]:
def mape(y_true: np.ndarray, y_pred: np.ndarray, min_casos: int = 5) -> float:
    """MAPE excluyendo observaciones con menos de min_casos (denominador inestable)."""
    mask = y_true >= min_casos
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


train = df_model[df_model['anio'] < 2024]
test  = df_model[df_model['anio'] == 2024]

X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET].values
y_baseline = test['casos_ma4'].values.ravel() # Ensure 1D array

# ── Baseline ──────────────────────────────────────────────────────────────────
mae_bl  = mean_absolute_error(y_test, y_baseline)
mape_bl = mape(y_test, y_baseline)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=300, max_depth=10,
                            min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train.values, y_train.values) # Convert to NumPy arrays
y_rf    = np.clip(rf.predict(X_test.values), 0, None)
mae_rf  = mean_absolute_error(y_test, y_rf)
mape_rf = mape(y_test, y_rf)

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    objective='reg:squarederror', random_state=42, n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train.values, y_train.values, # Convert to NumPy arrays
              eval_set=[(X_test.values, y_test)],
              verbose=False)
y_xgb    = np.clip(xgb_model.predict(X_test.values), 0, None)
mae_xgb  = mean_absolute_error(y_test, y_xgb)
mape_xgb = mape(y_test, y_xgb)

# Tabla comparativa
resultados = pd.DataFrame({
    'Modelo':   ['Baseline (MA4)', 'Random Forest', 'XGBoost'],
    'MAE':      [mae_bl,  mae_rf,  mae_xgb],
    'MAPE (%)': [mape_bl, mape_rf, mape_xgb],
    'Objetivo MAPE ≤ 30%': [
        '—',
        '✅' if mape_rf  <= 30 else '❌',
        '✅' if mape_xgb <= 30 else '❌',
    ]
})
print('=== Resultados en test set (brote 2024) ===')
print(resultados.to_string(index=False))

ValueError: Found input variables with inconsistent numbers of samples: [4818, 9636]

In [ ]:
wf_results = []
eval_years = [y for y in sorted(df_model.anio.unique()) if y >= 2020]

for pred_year in eval_years:
    tr = df_model[df_model['anio'] < pred_year]
    te = df_model[df_model['anio'] == pred_year]
    if len(tr) < 100 or len(te) == 0:
        continue

    # Random Forest
    m_rf = RandomForestRegressor(n_estimators=150, max_depth=8,
                                  min_samples_leaf=5, random_state=42, n_jobs=-1)
    m_rf.fit(tr[FEATURE_COLS].values, tr[TARGET].values) # Convert to NumPy arrays
    pred_rf = np.clip(m_rf.predict(te[FEATURE_COLS].values), 0, None)

    # XGBoost
    m_xgb = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
                               subsample=0.8, random_state=42, n_jobs=-1, verbosity=0)
    m_xgb.fit(tr[FEATURE_COLS].values, tr[TARGET].values) # Convert to NumPy arrays
    pred_xgb = np.clip(m_xgb.predict(te[FEATURE_COLS].values), 0, None)

    wf_results.append({
        'año':          pred_year,
        'n_obs':        len(te),
        'mape_baseline':mape(te[TARGET].values, te['casos_ma4'].values),
        'mape_rf':      mape(te[TARGET].values, pred_rf),
        'mape_xgb':     mape(te[TARGET].values, pred_xgb),
        'mae_rf':       mean_absolute_error(te[TARGET], pred_rf),
        'mae_xgb':      mean_absolute_error(te[TARGET], pred_xgb),
    })
    print(f'  {pred_year}: baseline {wf_results[-1]["mape_baseline"]:.1f}% | RF {wf_results[-1]["mape_rf"]:.1f}% | XGB {wf_results[-1]["mape_xgb"]:.1f}%')

df_wf = pd.DataFrame(wf_results)
df_wf['mejora_rf_pp']  = df_wf['mape_baseline'] - df_wf['mape_rf']
df_wf['mejora_xgb_pp'] = df_wf['mape_baseline'] - df_wf['mape_xgb']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(df_wf['año'], df_wf['mape_baseline'], 'o--', color='gray',   label='Baseline (MA4)', lw=1.5)
axes[0].plot(df_wf['año'], df_wf['mape_rf'],       'o-',  color=PALETTE[0], label='Random Forest', lw=2)
axes[0].plot(df_wf['año'], df_wf['mape_xgb'],      's-',  color=PALETTE[1], label='XGBoost',       lw=2)
axes[0].axhline(30, color=PALETTE[2], lw=1, ls=':', alpha=0.8, label='Objetivo ≤ 30%')
axes[0].set_xlabel('Año de predicción')
axes[0].set_ylabel('MAPE (%)')
axes[0].set_title('Validación walk-forward — MAPE por año')
axes[0].legend(fontsize=9)

x = np.arange(len(df_wf))
w = 0.35
axes[1].bar(x - w/2, df_wf['mejora_rf_pp'],  w, label='RF vs baseline',  color=PALETTE[0], alpha=0.85)
axes[1].bar(x + w/2, df_wf['mejora_xgb_pp'], w, label='XGB vs baseline', color=PALETTE[1], alpha=0.85)
axes[1].axhline(0, color='gray', lw=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_wf['año'])
axes[1].set_ylabel('Reducción MAPE (pp)')
axes[1].set_title('Mejora sobre baseline (pp)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Importancia de features — Random Forest
imp_rf = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

# Importancia XGBoost
imp_xgb = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp_rf.tail(15).plot(kind='barh', ax=axes[0], color=PALETTE[0], alpha=0.85)
axes[0].set_xlabel('Importancia (Gini)')
axes[0].set_title('Top 15 features — Random Forest')

imp_xgb.tail(15).plot(kind='barh', ax=axes[1], color=PALETTE[1], alpha=0.85)
axes[1].set_xlabel('Importancia (Gain)')
axes[1].set_title('Top 15 features — XGBoost')

plt.tight_layout()
plt.show()

print('Top 5 RF:',  imp_rf.tail(5).sort_values(ascending=False).index.tolist())
print('Top 5 XGB:', imp_xgb.tail(5).sort_values(ascending=False).index.tolist())

In [ ]:
# Predicción vs real — departamento de mayor carga 2024
deptos_2024 = test.groupby('departamento')[TARGET].sum().sort_values(ascending=False)
depto_foco  = deptos_2024.index[0]
print(f'Departamento con mayor carga en 2024: {depto_foco} ({deptos_2024.iloc[0]:.0f} casos)')

mask      = test['departamento'] == depto_foco
sem_d     = test.loc[mask, 'semana'].values
y_real_d  = y_test[mask.values]
y_rf_d    = y_rf[mask.values]
y_xgb_d   = y_xgb[mask.values]
y_base_d  = y_baseline[mask.values]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sem_d, y_real_d, 'o-', color='black',    lw=2,   label='Casos reales',  zorder=4)
ax.plot(sem_d, y_rf_d,   's--', color=PALETTE[0], lw=1.8, label='Random Forest', zorder=3)
ax.plot(sem_d, y_xgb_d,  'd-.', color=PALETTE[1], lw=1.8, label='XGBoost',       zorder=3)
ax.plot(sem_d, y_base_d, '^:',  color='gray',     lw=1.4, label='Baseline MA4',   zorder=2)
ax.fill_between(sem_d, y_xgb_d * 0.7, y_xgb_d * 1.3, alpha=0.1, color=PALETTE[1], label='XGB ±30%')
ax.set_xlabel('Semana epidemiológica')
ax.set_ylabel('Casos confirmados')
ax.set_title(f'{depto_foco} — predicción sobre brote 2024')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Resumen final del estado del proyecto
print('═' * 60)
print('RESUMEN FINAL — ESTADO DEL PROYECTO')
print('═' * 60)
print(f'Dataset SNVS (filas limpias)    : {len(df_master):,}')
print(f'Total casos 2018–2025           : {df_master.cantidad_casos.sum():,}')
print(f'Departamentos modelados         : {df_model.departamento.nunique()}')
print(f'Features construidas            : {len(FEATURE_COLS)}')
print()
print(f'Resultados en test set (2024):')
print(f'  Baseline (MA4)   MAPE = {mape_bl:.1f}%')
print(f'  Random Forest    MAPE = {mape_rf:.1f}%   (mejora: {mape_bl - mape_rf:+.1f} pp)')
print(f'  XGBoost          MAPE = {mape_xgb:.1f}%   (mejora: {mape_bl - mape_xgb:+.1f} pp)')
print()
print('Próximos pasos:')
print('  1. Conectar NASA POWER real (descomentar Opción A sección 7)')
print('  2. Incorporar NDVI via Google Earth Engine (Sentinel-2)')
print('  3. Agregar NBI y densidad del Censo 2022')
print('  4. Implementar corrección de subregistro (~1.5×)')
print('  5. Análisis regional NOA vs pampeana vs NEA (H3)')
print('  6. Evaluación de alerta binaria (F1, precisión) para operaciones')


In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['departamento', 'anio', 'semana']).copy()
    g  = df.groupby('departamento')

    # Lags epidemiológicos
    for lag in [1, 2, 3, 4]:
        df[f'casos_lag{lag}'] = g['cantidad_casos'].shift(lag)

    # Media móvil 4 SE (baseline)
    df['casos_ma4'] = g['cantidad_casos'].transform(
        lambda x: x.shift(1).rolling(4, min_periods=1).mean()
    )

    # Clima rezagado
    for var in ['temp_media', 'temp_max', 'humedad_media', 'precipitacion_mm']:
        if var not in df.columns:
            continue
        for lag in [2, 4, 6]:
            df[f'{var}_lag{lag}'] = g[var].shift(lag)

    if 'punto_rocio' in df.columns:
        df['punto_rocio_lag2'] = g['punto_rocio'].shift(2)
        df['punto_rocio_lag4'] = g['punto_rocio'].shift(4)

    if 'humedad_suelo' in df.columns:
        df['humedad_suelo_lag2'] = g['humedad_suelo'].shift(2)

    # Temperatura acumulada (GDD proxy)
    if 'temp_media' in df.columns:
        df['gdd_lag2'] = g['temp_media'].transform(
            lambda x: x.shift(2).rolling(3, min_periods=1).mean()
        )

    # Precipitación acumulada 3 SE
    if 'precipitacion_mm' in df.columns:
        df['precip_acum3_lag2'] = g['precipitacion_mm'].transform(
            lambda x: x.shift(2).rolling(3, min_periods=1).sum()
        )

    # Codificación cíclica de semana
    df['semana_sin'] = np.sin(2 * np.pi * df['semana'] / 52)
    df['semana_cos'] = np.cos(2 * np.pi * df['semana'] / 52)

    # Indicador de verano austral (SE 1–15 y SE 45–52)
    df['es_verano'] = df['semana'].apply(lambda w: 1 if (w <= 15 or w >= 45) else 0)

    # ID numérico de departamento
    df['depto_code'] = df['departamento'].astype('category').cat.codes.astype(int)

    return df

df_feat = build_features(df_modelado)

# Lista de features final (excluye columnas no disponibles)
CANDIDATES = [
    'casos_lag1','casos_lag2','casos_lag3','casos_lag4','casos_ma4',
    'temp_media_lag2','temp_media_lag4','temp_media_lag6',
    'temp_max_lag2','temp_max_lag4',
    'humedad_media_lag2','humedad_media_lag4',
    'precipitacion_mm_lag2','precipitacion_mm_lag4',
    'precip_acum3_lag2','gdd_lag2',
    'punto_rocio_lag2','punto_rocio_lag4',
    'humedad_suelo_lag2',
    'semana_sin','semana_cos','es_verano','depto_code',
]
FEATURE_COLS = [f for f in CANDIDATES if f in df_feat.columns]
TARGET = 'cantidad_casos'

df_model = df_feat[FEATURE_COLS + [TARGET, 'anio', 'semana', 'departamento', 'casos_ma4']].dropna()
print(f'✅ Dataset modelable: {len(df_model):,} filas | {len(FEATURE_COLS)} features')
print(f'Burn-in eliminado (NaN por lags): {len(df_feat) - len(df_model):,} filas')
print(f'Features: {FEATURE_COLS}')

In [ ]:
def mape(y_true: np.ndarray, y_pred: np.ndarray, min_casos: int = 5) -> float:
    """MAPE excluyendo observaciones con menos de min_casos (denominador inestable)."""
    mask = y_true >= min_casos
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


train = df_model[df_model['anio'] < 2024]
test  = df_model[df_model['anio'] == 2024]

X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET].values
y_baseline = test['casos_ma4'].values.ravel() # Ensure 1D array

# ── Baseline ──────────────────────────────────────────────────────────────────
mae_bl  = mean_absolute_error(y_test, y_baseline)
mape_bl = mape(y_test, y_baseline)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=300, max_depth=10,
                            min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train.values, y_train.values) # Convert to NumPy arrays
y_rf    = np.clip(rf.predict(X_test.values), 0, None)
mae_rf  = mean_absolute_error(y_test, y_rf)
mape_rf = mape(y_test, y_rf)

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    objective='reg:squarederror', random_state=42, n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train.values, y_train.values, # Convert to NumPy arrays
              eval_set=[(X_test.values, y_test)],
              verbose=False)
y_xgb    = np.clip(xgb_model.predict(X_test.values), 0, None)
mae_xgb  = mean_absolute_error(y_test, y_xgb)
mape_xgb = mape(y_test, y_xgb)

# Tabla comparativa
resultados = pd.DataFrame({
    'Modelo':   ['Baseline (MA4)', 'Random Forest', 'XGBoost'],
    'MAE':      [mae_bl,  mae_rf,  mae_xgb],
    'MAPE (%)': [mape_bl, mape_rf, mape_xgb],
    'Objetivo MAPE ≤ 30%': [
        '—',
        '✅' if mape_rf  <= 30 else '❌',
        '✅' if mape_xgb <= 30 else '❌',
    ]
})
print('=== Resultados en test set (brote 2024) ===')
print(resultados.to_string(index=False))

In [ ]:
wf_results = []
eval_years = [y for y in sorted(df_model.anio.unique()) if y >= 2020]

for pred_year in eval_years:
    tr = df_model[df_model['anio'] < pred_year]
    te = df_model[df_model['anio'] == pred_year]
    if len(tr) < 100 or len(te) == 0:
        continue

    # Random Forest
    m_rf = RandomForestRegressor(n_estimators=150, max_depth=8,
                                  min_samples_leaf=5, random_state=42, n_jobs=-1)
    m_rf.fit(tr[FEATURE_COLS].values, tr[TARGET].values) # Convert to NumPy arrays
    pred_rf = np.clip(m_rf.predict(te[FEATURE_COLS].values), 0, None)

    # XGBoost
    m_xgb = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
                               subsample=0.8, random_state=42, n_jobs=-1, verbosity=0)
    m_xgb.fit(tr[FEATURE_COLS].values, tr[TARGET].values) # Convert to NumPy arrays
    pred_xgb = np.clip(m_xgb.predict(te[FEATURE_COLS].values), 0, None)

    wf_results.append({
        'año':          pred_year,
        'n_obs':        len(te),
        'mape_baseline':mape(te[TARGET].values, te['casos_ma4'].values),
        'mape_rf':      mape(te[TARGET].values, pred_rf),
        'mape_xgb':     mape(te[TARGET].values, pred_xgb),
        'mae_rf':       mean_absolute_error(te[TARGET], pred_rf),
        'mae_xgb':      mean_absolute_error(te[TARGET], pred_xgb),
    })
    print(f'  {pred_year}: baseline {wf_results[-1]["mape_baseline"]:.1f}% | RF {wf_results[-1]["mape_rf"]:.1f}% | XGB {wf_results[-1]["mape_xgb"]:.1f}%')

df_wf = pd.DataFrame(wf_results)
df_wf['mejora_rf_pp']  = df_wf['mape_baseline'] - df_wf['mape_rf']
df_wf['mejora_xgb_pp'] = df_wf['mape_baseline'] - df_wf['mape_xgb']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(df_wf['año'], df_wf['mape_baseline'], 'o--', color='gray',   label='Baseline (MA4)', lw=1.5)
axes[0].plot(df_wf['año'], df_wf['mape_rf'],       'o-',  color=PALETTE[0], label='Random Forest', lw=2)
axes[0].plot(df_wf['año'], df_wf['mape_xgb'],      's-',  color=PALETTE[1], label='XGBoost',       lw=2)
axes[0].axhline(30, color=PALETTE[2], lw=1, ls=':', alpha=0.8, label='Objetivo ≤ 30%')
axes[0].set_xlabel('Año de predicción')
axes[0].set_ylabel('MAPE (%)')
axes[0].set_title('Validación walk-forward — MAPE por año')
axes[0].legend(fontsize=9)

x = np.arange(len(df_wf))
w = 0.35
axes[1].bar(x - w/2, df_wf['mejora_rf_pp'],  w, label='RF vs baseline',  color=PALETTE[0], alpha=0.85)
axes[1].bar(x + w/2, df_wf['mejora_xgb_pp'], w, label='XGB vs baseline', color=PALETTE[1], alpha=0.85)
axes[1].axhline(0, color='gray', lw=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_wf['año'])
axes[1].set_ylabel('Reducción MAPE (pp)')
axes[1].set_title('Mejora sobre baseline (pp)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Importancia de features — Random Forest
imp_rf = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

# Importancia XGBoost
imp_xgb = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp_rf.tail(15).plot(kind='barh', ax=axes[0], color=PALETTE[0], alpha=0.85)
axes[0].set_xlabel('Importancia (Gini)')
axes[0].set_title('Top 15 features — Random Forest')

imp_xgb.tail(15).plot(kind='barh', ax=axes[1], color=PALETTE[1], alpha=0.85)
axes[1].set_xlabel('Importancia (Gain)')
axes[1].set_title('Top 15 features — XGBoost')

plt.tight_layout()
plt.show()

print('Top 5 RF:',  imp_rf.tail(5).sort_values(ascending=False).index.tolist())
print('Top 5 XGB:', imp_xgb.tail(5).sort_values(ascending=False).index.tolist())

In [ ]:
# Predicción vs real — departamento de mayor carga 2024
deptos_2024 = test.groupby('departamento')[TARGET].sum().sort_values(ascending=False)
depto_foco  = deptos_2024.index[0]
print(f'Departamento con mayor carga en 2024: {depto_foco} ({deptos_2024.iloc[0]:.0f} casos)')

mask      = test['departamento'] == depto_foco
sem_d     = test.loc[mask, 'semana'].values
y_real_d  = y_test[mask.values]
y_rf_d    = y_rf[mask.values]
y_xgb_d   = y_xgb[mask.values]
y_base_d  = y_baseline[mask.values]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sem_d, y_real_d, 'o-', color='black',    lw=2,   label='Casos reales',  zorder=4)
ax.plot(sem_d, y_rf_d,   's--', color=PALETTE[0], lw=1.8, label='Random Forest', zorder=3)
ax.plot(sem_d, y_xgb_d,  'd-.', color=PALETTE[1], lw=1.8, label='XGBoost',       zorder=3)
ax.plot(sem_d, y_base_d, '^:',  color='gray',     lw=1.4, label='Baseline MA4',   zorder=2)
ax.fill_between(sem_d, y_xgb_d * 0.7, y_xgb_d * 1.3, alpha=0.1, color=PALETTE[1], label='XGB ±30%')
ax.set_xlabel('Semana epidemiológica')
ax.set_ylabel('Casos confirmados')
ax.set_title(f'{depto_foco} — predicción sobre brote 2024')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Resumen final del estado del proyecto
print('═' * 60)
print('RESUMEN FINAL — ESTADO DEL PROYECTO')
print('═' * 60)
print(f'Dataset SNVS (filas limpias)    : {len(df_master):,}')
print(f'Total casos 2018–2025           : {df_master.cantidad_casos.sum():,}')
print(f'Departamentos modelados         : {df_model.departamento.nunique()}')
print(f'Features construidas            : {len(FEATURE_COLS)}')
print()
print(f'Resultados en test set (2024):')
print(f'  Baseline (MA4)   MAPE = {mape_bl:.1f}%')
print(f'  Random Forest    MAPE = {mape_rf:.1f}%   (mejora: {mape_bl - mape_rf:+.1f} pp)')
print(f'  XGBoost          MAPE = {mape_xgb:.1f}%   (mejora: {mape_bl - mape_xgb:+.1f} pp)')
print()
print('Próximos pasos:')
print('  1. Conectar NASA POWER real (descomentar Opción A sección 7)')
print('  2. Incorporar NDVI via Google Earth Engine (Sentinel-2)')
print('  3. Agregar NBI y densidad del Censo 2022')
print('  4. Implementar corrección de subregistro (~1.5×)')
print('  5. Análisis regional NOA vs pampeana vs NEA (H3)')
print('  6. Evaluación de alerta binaria (F1, precisión) para operaciones')


In [ ]:
# ── 8.1 Limpieza del dataset SNVS ─────────────────────────────────────────────
print('=== REPORTE DE CALIDAD — SNVS ===')
print(f'Filas originales         : {len(df_raw):,}')
print(f'Nulos en cantidad_casos  : {df_raw.cantidad_casos.isna().sum()}')
print(f'Departamentos únicos     : {df_raw.departamento.nunique()}')
print(f'Rango de años            : {df_raw.anio.min()} – {df_raw.anio.max()}')
print()

# Decisión 1: excluir registros sin departamento identificable
# Justificación: no pueden asignarse geográficamente para el modelo espacial
mask_desconocido = df_raw['departamento'].str.lower().str.strip().isin(
    ['desconocido', '*sin dato*', 'sin dato', 'nd', '']
)
n_desconocido = mask_desconocido.sum()
print(f'Filas con departamento desconocido: {n_desconocido:,} → EXCLUIDAS')

# Decisión 2: excluir años fuera del rango de interés
mask_anio = (df_raw['anio'] < 2018) | (df_raw['anio'] > 2025)
n_anio = mask_anio.sum()
print(f'Filas con año fuera de 2018–2025 : {n_anio:,} → EXCLUIDAS')

# Decisión 3: corregir semana 53 → 52 (artefacto de calendario ISO)
n_se53 = (df_raw['semana'] == 53).sum()
df_raw.loc[df_raw['semana'] == 53, 'semana'] = 52
print(f'Filas con SE=53 corregidas a 52  : {n_se53:,}')

# Aplicar filtros
df_snvs = df_raw[~mask_desconocido & ~mask_anio].copy()

# Decisión 4: los duplicados del archivo 2025 (primer y segundo semestre)
# pueden solaparse en SE 1–15. Deduplicar por (anio, semana, departamento).
n_antes = len(df_snvs)
df_snvs = df_snvs.groupby(
    ['anio', 'semana', 'departamento', 'provincia'], as_index=False
)['cantidad_casos'].sum()
n_despues = len(df_snvs)
print(f'Filas después de deduplicación   : {n_despues:,} (antes: {n_antes:,})')

# Reporte final
print(f'\n✅ Dataset SNVS limpio           : {len(df_snvs):,} filas | {df_snvs.cantidad_casos.sum():,} casos')
print(f'   Departamentos únicos          : {df_snvs.departamento.nunique()}')
print(f'   Provincias                    : {df_snvs.provincia.nunique()}')

In [ ]:
# ── 8.2 Limpieza del dataset climático ────────────────────────────────────────
print('=== REPORTE DE CALIDAD — CLIMA ===')
print(f'Filas originales   : {len(df_clima):,}')
print(f'Nulos por columna:')
nulos = df_clima.isnull().sum()
print(nulos[nulos > 0].to_string() if (nulos > 0).any() else '  Ninguno')

# Decisión: valores -999 de NASA POWER (missing) → interpolación lineal por provincia
clima_vars_num = df_clima.select_dtypes(include='number').columns.difference(['anio', 'semana'])
for var in clima_vars_num:
    n_miss = df_clima[var].isna().sum()
    if n_miss > 0:
        df_clima[var] = (df_clima.groupby('provincia')[var]
                                  .transform(lambda x: x.interpolate(method='linear', limit_direction='both')))
        print(f'  Interpolados {n_miss} valores en {var}')

print(f'\n✅ Dataset climático limpio: {len(df_clima):,} filas')

In [ ]:
# ── 8.3 Join epidemiológico × climático ───────────────────────────────────────

# Normalizar nombres de provincia para el join
PROV_MAP = {
    'Ciudad De Buenos Aires': 'CABA',
    'Ciudad de Buenos Aires': 'CABA',
    'Caba': 'CABA',
    'Entre Rios': 'Entre Ríos',
    'Cordoba': 'Córdoba',
    'Tucuman': 'Tucumán',
    'Santiago Del Estero': 'Santiago del Estero',
}
df_snvs['provincia_norm'] = (df_snvs['provincia'].str.strip()
                                                  .str.title()
                                                  .replace(PROV_MAP))

df_master = df_snvs.merge(
    df_clima.rename(columns={'provincia': 'provincia_norm'}),
    on=['anio', 'semana', 'provincia_norm'],
    how='left'
)

cobertura_clima = df_master['temp_media'].notna().mean() * 100
print(f'✅ Dataset maestro: {len(df_master):,} filas')
print(f'   Cobertura climática: {cobertura_clima:.1f}% de las filas')
print(f'   Columnas: {list(df_master.columns)}')
df_master.head(3)

In [ ]:
# ── 8.4 Reporte de calidad final ─────────────────────────────────────────────
print('=' * 55)
print('REPORTE DE CALIDAD — DATASET MAESTRO FINAL')
print('=' * 55)
print(f'Filas totales               : {len(df_master):,}')
print(f'Columnas                    : {df_master.shape[1]}')
print(f'Años cubiertos              : {sorted(df_master.anio.unique())}')
print(f'Departamentos únicos        : {df_master.departamento.nunique()}')
print(f'Provincias                  : {df_master.provincia.nunique()}')
print(f'Total casos confirmados     : {df_master.cantidad_casos.sum():,}')
print(f'Semanas epidemiológicas     : {df_master.semana.min()} – {df_master.semana.max()}')
print()
print('Estadísticas de casos por departamento-semana:')
print(df_master['cantidad_casos'].describe().to_string())
print()
print('Nulos por columna:')
nulos_final = df_master.isnull().sum()
print(nulos_final[nulos_final > 0].to_string() if (nulos_final > 0).any() else '  Ninguno')

In [ ]:
# Exportar dataset completo
ruta_completo = OUT_DIR / 'dengue_dataset_limpio.csv'
df_master.to_csv(ruta_completo, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_completo}  ({len(df_master):,} filas)')

# Subconjunto: departamentos con suficiente historia
conteo_depto = df_master.groupby('departamento').size()
deptos_suficientes = conteo_depto[conteo_depto >= 30].index
df_modelado = df_master[df_master.departamento.isin(deptos_suficientes)].copy()

ruta_modelado = OUT_DIR / 'dengue_dataset_departamentos_top.csv'
df_modelado.to_csv(ruta_modelado, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_modelado}  ({len(df_modelado):,} filas | {df_modelado.departamento.nunique()} departamentos)')

# Diccionario de variables
diccionario = pd.DataFrame([
    {'variable': 'anio',              'tipo': 'int',   'descripcion': 'Año calendario'},
    {'variable': 'semana',            'tipo': 'int',   'descripcion': 'Semana epidemiológica ISO (1–52)'},
    {'variable': 'departamento',      'tipo': 'str',   'descripcion': 'Nombre del departamento/partido'},
    {'variable': 'provincia',         'tipo': 'str',   'descripcion': 'Nombre de la provincia'},
    {'variable': 'cantidad_casos',    'tipo': 'int',   'descripcion': 'Casos confirmados de dengue (TARGET)'},
    {'variable': 'temp_media',        'tipo': 'float', 'descripcion': 'Temperatura media 2m (°C) — NASA POWER T2M'},
    {'variable': 'temp_max',          'tipo': 'float', 'descripcion': 'Temperatura máxima 2m (°C) — NASA POWER T2M_MAX'},
    {'variable': 'temp_min',          'tipo': 'float', 'descripcion': 'Temperatura mínima 2m (°C) — NASA POWER T2M_MIN'},
    {'variable': 'temp_rango',        'tipo': 'float', 'descripcion': 'Amplitud térmica diaria (°C) — NASA POWER T2M_RANGE'},
    {'variable': 'humedad_media',     'tipo': 'float', 'descripcion': 'Humedad relativa media 2m (%) — NASA POWER RH2M'},
    {'variable': 'humedad_max',       'tipo': 'float', 'descripcion': 'Humedad relativa máxima (%)'},
    {'variable': 'humedad_min',       'tipo': 'float', 'descripcion': 'Humedad relativa mínima (%)'},
    {'variable': 'precipitacion_mm',  'tipo': 'float', 'descripcion': 'Precipitación diaria corregida (mm) — NASA POWER PRECTOTCORR'},
    {'variable': 'precipitacion_acum','tipo': 'float', 'descripcion': 'Precipitación acumulada semanal (mm)'},
    {'variable': 'radiacion_solar',   'tipo': 'float', 'descripcion': 'Radiación solar incidente (W/m²) — NASA POWER ALLSKY_SFC_SW_DWN'},
    {'variable': 'viento_medio',      'tipo': 'float', 'descripcion': 'Velocidad del viento media 2m (m/s) — NASA POWER WS2M'},
    {'variable': 'humedad_suelo',     'tipo': 'float', 'descripcion': 'Humedad del suelo capa raíz — NASA POWER GWETROOT'},
    {'variable': 'punto_rocio',       'tipo': 'float', 'descripcion': 'Temperatura de punto de rocío (°C) — NASA POWER T2MDEW'},
])

ruta_diccionario = OUT_DIR / 'diccionario_variables.csv'
diccionario.to_csv(ruta_diccionario, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_diccionario}')

In [ ]:
# 10.1 Serie temporal nacional
serie = df_master.groupby(['anio', 'semana'])['cantidad_casos'].sum().reset_index()
serie['t'] = (serie['anio'] - 2018) * 52 + serie['semana']

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(serie['t'], serie['cantidad_casos'], alpha=0.2, color=PALETTE[0])
ax.plot(serie['t'], serie['cantidad_casos'], color=PALETTE[0], lw=1.2)
ax.axvspan(6 * 52, 7 * 52, alpha=0.07, color='red', label='Brote 2024 (test set)')
xticks = [i * 52 + 1 for i in range(8)]
ax.set_xticks(xticks)
ax.set_xticklabels(range(2018, 2026), fontsize=9)
ax.set_xlabel('Año')
ax.set_ylabel('Casos confirmados')
ax.set_title('Casos de dengue — suma semanal nacional (SNVS 2018–2025)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(df_master.groupby('anio')['cantidad_casos'].sum().rename('casos_totales').to_frame())

In [ ]:
# 10.2 Estacionalidad + top departamentos + distribución
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Estacionalidad (sin 2024 para no distorsionar)
estac = df_master[df_master.anio < 2024].groupby('semana')['cantidad_casos'].mean()
axes[0].bar(estac.index, estac.values, color=PALETTE[0], alpha=0.8, width=0.9)
axes[0].set_title('Estacionalidad media (2018–2023)')
axes[0].set_xlabel('Semana epidemiológica')
axes[0].set_ylabel('Casos promedio')

# Top 12 departamentos
top_d = (df_master[~df_master.departamento.str.lower().isin(['desconocido'])]
               .groupby('departamento')['cantidad_casos']
               .sum().sort_values(ascending=True).tail(12))
top_d.plot(kind='barh', ax=axes[1], color=PALETTE[1], alpha=0.85)
axes[1].set_title('Top 12 departamentos (2018–2025)')
axes[1].set_xlabel('Casos totales')
axes[1].tick_params(axis='y', labelsize=8)

# Distribución log
vals = df_master[df_master.cantidad_casos > 0]['cantidad_casos']
axes[2].hist(np.log1p(vals), bins=50, color=PALETTE[2], alpha=0.8, edgecolor='white')
axes[2].set_title('Distribución de casos (log+1)')
axes[2].set_xlabel('log(casos + 1)')
axes[2].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

In [ ]:
# 10.3 Heatmap año × semana
pivot = (df_master.groupby(['anio', 'semana'])['cantidad_casos']
               .sum().unstack('semana').fillna(0))

fig, ax = plt.subplots(figsize=(15, 4))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0, cbar_kws={'label': 'Casos'})
ax.set_title('Heatmap casos — suma nacional por año y semana epidemiológica')
ax.set_xlabel('Semana epidemiológica')
ax.set_ylabel('Año')
plt.tight_layout()
plt.show()

In [ ]:
# 10.4 Correlación clima × casos por lag (hipótesis H2)
lag_range  = range(0, 9)
clima_corr = ['temp_media', 'humedad_media', 'precipitacion_mm', 'punto_rocio']
clima_corr = [v for v in clima_corr if v in df_master.columns]

df_sorted = df_master.sort_values(['departamento', 'anio', 'semana'])
corr_df   = pd.DataFrame(index=lag_range, columns=clima_corr, dtype=float)

for lag in lag_range:
    for var in clima_corr:
        shifted = df_sorted.groupby('departamento')[var].shift(lag)
        corr_df.loc[lag, var] = shifted.corr(df_sorted['cantidad_casos'])

fig, ax = plt.subplots(figsize=(9, 4))
for i, var in enumerate(clima_corr):
    ax.plot(corr_df.index, corr_df[var].astype(float), marker='o',
            label=var, color=PALETTE[i], lw=1.8)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Lag (semanas)')
ax.set_ylabel('Correlación de Pearson')
ax.set_title('Correlación climática vs casos — por lag temporal\n(prueba de H2: temperatura como predictor dominante)')
ax.legend(fontsize=9)
ax.set_xticks(list(lag_range))
plt.tight_layout()
plt.show()

lag_optimo = corr_df['temp_media'].astype(float).idxmax()
print(f'Lag óptimo para temp_media: {lag_optimo} semanas (correlación = {corr_df.loc[lag_optimo, "temp_media"]:.3f})')

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['departamento', 'anio', 'semana']).copy()
    g  = df.groupby('departamento')

    # Lags epidemiológicos
    for lag in [1, 2, 3, 4]:
        df[f'casos_lag{lag}'] = g['cantidad_casos'].shift(lag)

    # Media móvil 4 SE (baseline)
    df['casos_ma4'] = g['cantidad_casos'].transform(
        lambda x: x.shift(1).rolling(4, min_periods=1).mean()
    )

    # Clima rezagado
    for var in ['temp_media', 'temp_max', 'humedad_media', 'precipitacion_mm']:
        if var not in df.columns:
            continue
        for lag in [2, 4, 6]:
            df[f'{var}_lag{lag}'] = g[var].shift(lag)

    if 'punto_rocio' in df.columns:
        df['punto_rocio_lag2'] = g['punto_rocio'].shift(2)
        df['punto_rocio_lag4'] = g['punto_rocio'].shift(4)

    if 'humedad_suelo' in df.columns:
        df['humedad_suelo_lag2'] = g['humedad_suelo'].shift(2)

    # Temperatura acumulada (GDD proxy)
    if 'temp_media' in df.columns:
        df['gdd_lag2'] = g['temp_media'].transform(
            lambda x: x.shift(2).rolling(3, min_periods=1).mean()
        )

    # Precipitación acumulada 3 SE
    if 'precipitacion_mm' in df.columns:
        df['precip_acum3_lag2'] = g['precipitacion_mm'].transform(
            lambda x: x.shift(2).rolling(3, min_periods=1).sum()
        )

    # Codificación cíclica de semana
    df['semana_sin'] = np.sin(2 * np.pi * df['semana'] / 52)
    df['semana_cos'] = np.cos(2 * np.pi * df['semana'] / 52)

    # Indicador de verano austral (SE 1–15 y SE 45–52)
    df['es_verano'] = df['semana'].apply(lambda w: 1 if (w <= 15 or w >= 45) else 0)

    # ID numérico de departamento
    df['depto_code'] = df['departamento'].astype('category').cat.codes.astype(int)

    return df

df_feat = build_features(df_modelado)

# Lista de features final (excluye columnas no disponibles)
CANDIDATES = [
    'casos_lag1','casos_lag2','casos_lag3','casos_lag4','casos_ma4',
    'temp_media_lag2','temp_media_lag4','temp_media_lag6',
    'temp_max_lag2','temp_max_lag4',
    'humedad_media_lag2','humedad_media_lag4',
    'precipitacion_mm_lag2','precipitacion_mm_lag4',
    'precip_acum3_lag2','gdd_lag2',
    'punto_rocio_lag2','punto_rocio_lag4',
    'humedad_suelo_lag2',
    'semana_sin','semana_cos','es_verano','depto_code',
]
FEATURE_COLS = [f for f in CANDIDATES if f in df_feat.columns]
TARGET = 'cantidad_casos'

df_model = df_feat[FEATURE_COLS + [TARGET, 'anio', 'semana', 'departamento', 'casos_ma4']].dropna()
print(f'✅ Dataset modelable: {len(df_model):,} filas | {len(FEATURE_COLS)} features')
print(f'   Burn-in eliminado (NaN por lags): {len(df_feat) - len(df_model):,} filas')
print(f'   Features: {FEATURE_COLS})

In [ ]:
def mape(y_true: np.ndarray, y_pred: np.ndarray, min_casos: int = 5) -> float:
    """MAPE excluyendo observaciones con menos de min_casos (denominador inestable)."""
    mask = y_true >= min_casos
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


train = df_model[df_model['anio'] < 2024]
test  = df_model[df_model['anio'] == 2024]

X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET].values
y_baseline = test['casos_ma4'].values.ravel() # Ensure 1D array

# ── Baseline ──────────────────────────────────────────────────────────────────
mae_bl  = mean_absolute_error(y_test, y_baseline)
mape_bl = mape(y_test, y_baseline)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=300, max_depth=10,
                            min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train.values, y_train.values) # Convert to NumPy arrays
y_rf    = np.clip(rf.predict(X_test.values), 0, None)
mae_rf  = mean_absolute_error(y_test, y_rf)
mape_rf = mape(y_test, y_rf)

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    objective='reg:squarederror', random_state=42, n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train.values, y_train.values, # Convert to NumPy arrays
              eval_set=[(X_test.values, y_test)],
              verbose=False)
y_xgb    = np.clip(xgb_model.predict(X_test.values), 0, None)
mae_xgb  = mean_absolute_error(y_test, y_xgb)
mape_xgb = mape(y_test, y_xgb)

# Tabla comparativa
resultados = pd.DataFrame({
    'Modelo':   ['Baseline (MA4)', 'Random Forest', 'XGBoost'],
    'MAE':      [mae_bl,  mae_rf,  mae_xgb],
    'MAPE (%)': [mape_bl, mape_rf, mape_xgb],
    'Objetivo MAPE ≤ 30%': [
        '—',
        '✅' if mape_rf  <= 30 else '❌',
        '✅' if mape_xgb <= 30 else '❌',
    ]
})
print('=== Resultados en test set (brote 2024) ===')
print(resultados.to_string(index=False))

In [ ]:
wf_results = []
eval_years = [y for y in sorted(df_model.anio.unique()) if y >= 2020]

for pred_year in eval_years:
    tr = df_model[df_model['anio'] < pred_year]
    te = df_model[df_model['anio'] == pred_year]
    if len(tr) < 100 or len(te) == 0:
        continue

    # Random Forest
    m_rf = RandomForestRegressor(n_estimators=150, max_depth=8,
                                  min_samples_leaf=5, random_state=42, n_jobs=-1)
    m_rf.fit(tr[FEATURE_COLS].values, tr[TARGET].values) # Convert to NumPy arrays
    pred_rf = np.clip(m_rf.predict(te[FEATURE_COLS].values), 0, None)

    # XGBoost
    m_xgb = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
                               subsample=0.8, random_state=42, n_jobs=-1, verbosity=0)
    m_xgb.fit(tr[FEATURE_COLS].values, tr[TARGET].values) # Convert to NumPy arrays
    pred_xgb = np.clip(m_xgb.predict(te[FEATURE_COLS].values), 0, None)

    wf_results.append({
        'año':          pred_year,
        'n_obs':        len(te),
        'mape_baseline':mape(te[TARGET].values, te['casos_ma4'].values),
        'mape_rf':      mape(te[TARGET].values, pred_rf),
        'mape_xgb':     mape(te[TARGET].values, pred_xgb),
        'mae_rf':       mean_absolute_error(te[TARGET], pred_rf),
        'mae_xgb':      mean_absolute_error(te[TARGET], pred_xgb),
    })
    print(f'  {pred_year}: baseline {wf_results[-1]["mape_baseline"]:.1f}% | RF {wf_results[-1]["mape_rf"]:.1f}% | XGB {wf_results[-1]["mape_xgb"]:.1f}%')

df_wf = pd.DataFrame(wf_results)
df_wf['mejora_rf_pp']  = df_wf['mape_baseline'] - df_wf['mape_rf']
df_wf['mejora_xgb_pp'] = df_wf['mape_baseline'] - df_wf['mape_xgb']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(df_wf['año'], df_wf['mape_baseline'], 'o--', color='gray',   label='Baseline (MA4)', lw=1.5)
axes[0].plot(df_wf['año'], df_wf['mape_rf'],       'o-',  color=PALETTE[0], label='Random Forest', lw=2)
axes[0].plot(df_wf['año'], df_wf['mape_xgb'],      's-',  color=PALETTE[1], label='XGBoost',       lw=2)
axes[0].axhline(30, color=PALETTE[2], lw=1, ls=':', alpha=0.8, label='Objetivo ≤ 30%')
axes[0].set_xlabel('Año de predicción')
axes[0].set_ylabel('MAPE (%)')
axes[0].set_title('Validación walk-forward — MAPE por año')
axes[0].legend(fontsize=9)

x = np.arange(len(df_wf))
w = 0.35
axes[1].bar(x - w/2, df_wf['mejora_rf_pp'],  w, label='RF vs baseline',  color=PALETTE[0], alpha=0.85)
axes[1].bar(x + w/2, df_wf['mejora_xgb_pp'], w, label='XGB vs baseline', color=PALETTE[1], alpha=0.85)
axes[1].axhline(0, color='gray', lw=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_wf['año'])
axes[1].set_ylabel('Reducción MAPE (pp)')
axes[1].set_title('Mejora sobre baseline (pp)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Importancia de features — Random Forest
imp_rf = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

# Importancia XGBoost
imp_xgb = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp_rf.tail(15).plot(kind='barh', ax=axes[0], color=PALETTE[0], alpha=0.85)
axes[0].set_xlabel('Importancia (Gini)')
axes[0].set_title('Top 15 features — Random Forest')

imp_xgb.tail(15).plot(kind='barh', ax=axes[1], color=PALETTE[1], alpha=0.85)
axes[1].set_xlabel('Importancia (Gain)')
axes[1].set_title('Top 15 features — XGBoost')

plt.tight_layout()
plt.show()

print('Top 5 RF:',  imp_rf.tail(5).sort_values(ascending=False).index.tolist())
print('Top 5 XGB:', imp_xgb.tail(5).sort_values(ascending=False).index.tolist())

In [ ]:
# Predicción vs real — departamento de mayor carga 2024
deptos_2024 = test.groupby('departamento')[TARGET].sum().sort_values(ascending=False)
depto_foco  = deptos_2024.index[0]
print(f'Departamento con mayor carga en 2024: {depto_foco} ({deptos_2024.iloc[0]:.0f} casos)')

mask      = test['departamento'] == depto_foco
sem_d     = test.loc[mask, 'semana'].values
y_real_d  = y_test[mask.values]
y_rf_d    = y_rf[mask.values]
y_xgb_d   = y_xgb[mask.values]
y_base_d  = y_baseline[mask.values]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sem_d, y_real_d, 'o-', color='black',    lw=2,   label='Casos reales',  zorder=4)
ax.plot(sem_d, y_rf_d,   's--', color=PALETTE[0], lw=1.8, label='Random Forest', zorder=3)
ax.plot(sem_d, y_xgb_d,  'd-.', color=PALETTE[1], lw=1.8, label='XGBoost',       zorder=3)
ax.plot(sem_d, y_base_d, '^:',  color='gray',     lw=1.4, label='Baseline MA4',   zorder=2)
ax.fill_between(sem_d, y_xgb_d * 0.7, y_xgb_d * 1.3, alpha=0.1, color=PALETTE[1], label='XGB ±30%')
ax.set_xlabel('Semana epidemiológica')
ax.set_ylabel('Casos confirmados')
ax.set_title(f'{depto_foco} — predicción sobre brote 2024')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Resumen final del estado del proyecto
print('═' * 60)
print('RESUMEN FINAL — ESTADO DEL PROYECTO')
print('═' * 60)
print(f'Dataset SNVS (filas limpias)    : {len(df_master):,}')
print(f'Total casos 2018–2025           : {df_master.cantidad_casos.sum():,}')
print(f'Departamentos modelados         : {df_model.departamento.nunique()}')
print(f'Features construidas            : {len(FEATURE_COLS)}')
print()
print(f'Resultados en test set (2024):')
print(f'  Baseline (MA4)   MAPE = {mape_bl:.1f}%')
print(f'  Random Forest    MAPE = {mape_rf:.1f}%   (mejora: {mape_bl - mape_rf:+.1f} pp)')
print(f'  XGBoost          MAPE = {mape_xgb:.1f}%   (mejora: {mape_bl - mape_xgb:+.1f} pp)')
print()
print('Próximos pasos:')
print('  1. Conectar NASA POWER real (descomentar Opción A sección 7)')
print('  2. Incorporar NDVI via Google Earth Engine (Sentinel-2)')
print('  3. Agregar NBI y densidad del Censo 2022')
print('  4. Implementar corrección de subregistro (~1.5×)')
print('  5. Análisis regional NOA vs pampeana vs NEA (H3)')
print('  6. Evaluación de alerta binaria (F1, precisión) para operaciones')


In [ ]:
# ── OPCIÓN B: Datos sintéticos calibrados (sin internet) ──────────────────────
# Misma estructura que los datos reales de NASA POWER.
# Reemplazar por Opción A en producción.

np.random.seed(42)
clima_rows = []
for anio in range(2018, 2026):
    for semana in range(1, 53):
        for prov, (lat, lon) in CENTROIDES_PROV.items():
            angle   = 2 * np.pi * (semana - 1) / 52
            t_base  = 22 - abs(lat + 32) * 0.5
            t_media = t_base + 8 * np.cos(angle + np.pi) + np.random.normal(0, 1.5)
            hum      = float(np.clip(65 + 15 * np.cos(angle + np.pi) + np.random.normal(0, 5), 30, 99))
            precip  = max(0.0, 40 * np.cos(angle + np.pi/4) + 20 + np.random.exponential(15))
            clima_rows.append({
                'anio': anio, 'semana': semana, 'provincia': prov,
                'temp_media':       round(t_media, 2),
                'temp_max':         round(t_media + 5 + np.random.normal(0, 1), 2),
                'temp_min':         round(t_media - 5 + np.random.normal(0, 1), 2),
                'temp_rango':       round(10 + np.random.exponential(2), 2),
                'humedad_media':    round(hum, 2),
                'humedad_max':      round(min(hum + 10, 99), 2),
                'humedad_min':      round(max(hum - 15, 20), 2),
                'precipitacion_mm': round(precip, 2),
                'precipitacion_acum': round(precip * 7, 2),
                'radiacion_solar':  round(max(0, 200 + 150 * np.cos(angle + np.pi) + np.random.normal(0, 30)), 2),
                'viento_medio':     round(max(0, 3 + np.random.exponential(1.5)), 2),
                'humedad_suelo':    round(float(np.clip(0.3 + 0.15 * np.cos(angle + np.pi/4) + np.random.normal(0, 0.05), 0.05, 0.9)), 3),
                'punto_rocio':      round(t_media - 5 + np.random.normal(0, 2), 2),
            })

df_clima = pd.DataFrame(clima_rows)
print(f'✅ Datos climáticos: {len(df_clima):,} filas — {df_clima.provincia.nunique()} provincias × {df_clima.anio.nunique()} años')
print('⚠️  Actualmente usando datos sintéticos. Descomentar Opción A para datos reales de NASA POWER.')

In [ ]:
# ── 6 parámetros seleccionados (para evitar errores 422 de la API) ─────────────
NASA_PARAMS = ','.join([
    # Temperatura (3)
    'T2M', 'T2M_MAX', 'T2M_MIN',
    # Agua y humedad (2)
    'PRECTOTCORR', 'RH2M',
    # Viento (1)
    'WS2M'
])

# Centroides por provincia (lat, lon)
CENTROIDES_PROV = {
    'Buenos Aires':        (-36.60, -60.00),
    'CABA':                (-34.61, -58.44),
    'Córdoba':             (-31.42, -64.18),
    'Santa Fe':            (-31.63, -60.70),
    'Tucumán':             (-26.82, -65.22),
    'Salta':               (-24.79, -65.41),
    'Misiones':            (-27.36, -55.90),
    'Jujuy':               (-24.18, -65.30),
    'Entre Ríos':          (-31.74, -60.54),
    'Chaco':               (-27.46, -58.99),
    'Formosa':             (-26.18, -58.18),
    'Corrientes':          (-27.47, -58.83),
    'Santiago del Estero': (-27.78, -64.26),
}

print(f'Parámetros a descargar: {len(NASA_PARAMS.split(","))} variables')
print(f'Provincias: {len(CENTROIDES_PROV)}')


In [ ]:
def fetch_nasa_power(lat: float, lon: float,
                     start_date: str = '20180101',
                     end_date:   str = '20251231',
                     retries: int = 3) -> pd.DataFrame:
    """
    Descarga datos climáticos DIARIOS de NASA POWER para una coordenada.
    Formato CSV — evita pivoteo de JSON. Sin API key.

    Parámetros
    ----------
    lat, lon    : coordenadas del centroide
    start_date  : fecha inicio YYYYMMDD
    end_date    : fecha fin   YYYYMMDD
    retries     : intentos ante errores de red

    Retorna
    -------
    DataFrame con columnas: fecha, anio, semana + variables climáticas
    """
    url = (
        f'https://power.larc.nasa.gov/api/temporal/daily/point'
        f'?parameters={NASA_PARAMS}'
        f'&community=SB'
        f'&longitude={lon}'
        f'&latitude={lat}'
        f'&start={start_date}'
        f'&end={end_date}'
        f'&format=CSV'
    )

    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=120)
            resp.raise_for_status()
            break
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f'    Reintento {attempt + 1}/{retries}...')
            time.sleep(5)

    # La API agrega cabeceras de metadatos antes del CSV real.
    # Saltamos hasta encontrar la línea con los nombres de columna (empieza con 'YEAR').
    lines = resp.text.splitlines()
    try:
        header_idx = next(i for i, line in enumerate(lines) if line.startswith('YEAR'))
    except StopIteration:
        raise ValueError('Respuesta inesperada de NASA POWER — no se encontró encabezado CSV')

    df = pd.read_csv(
        StringIO('\n'.join(lines[header_idx:])),
        na_values='-999',
    )

    # Construir fecha y semana epidemiológica de forma robusta
    if 'YEAR' in df.columns and 'DOY' in df.columns:
        df['fecha'] = pd.to_datetime(
            df['YEAR'].astype(str) + df['DOY'].astype(str).str.zfill(3),
            format='%Y%j'
        )
    elif 'YEAR' in df.columns and 'MO' in df.columns and 'DY' in df.columns:
        df['fecha'] = pd.to_datetime(
            df['YEAR'].astype(str) + '-' + df['MO'].astype(str) + '-' + df['DY'].astype(str),
            format='%Y-%m-%d'
        )
    else:
        raise ValueError(
            "NASA POWER response missing sufficient date columns (neither DOY nor MO/DY found).")

    df['anio']   = df['fecha'].dt.isocalendar().year.astype(int)
    df['semana'] = df['fecha'].dt.isocalendar().week.astype(int)

    # Renombrar a nombres canónicos (se han simplificado algunos, mantener los que existen)
    df = df.rename(columns={
        'T2M':              'temp_media',
        'T2M_MAX':          'temp_max',
        'T2M_MIN':          'temp_min',
        'PRECTOTCORR':      'precipitacion_mm',
        'RH2M':             'humedad_media',
        'WS2M':             'viento_medio',
        'T2M_RANGE':        'temp_rango',
        'T_WET_RANGE_2M':   'temp_bulbo_humedo_rango',
        'TS':               'temp_superficie',
        'PRECTOTCORR_SUM':  'precipitacion_acum',
        'RH2M_MAX':         'humedad_max',
        'RH2M_MIN':         'humedad_min',
        'QV2M':             'humedad_especifica',
        'ALLSKY_SFC_SW_DWN':'radiacion_solar',
        'ALLSKY_SFC_LW_DWN':'radiacion_onda_larga',
        'WS2M_MAX':         'viento_max',
        'PS':               'presion',
        'GWETROOT':         'humedad_suelo',
        'EVPTRNS':          'evapotranspiracion',
        'T2MDEW':           'punto_rocio',
    })

    # Agregar de diario a semanal (media; precipitación como suma)
    # Asegurarse de que solo se usen las columnas presentes en el DataFrame
    clima_vars_all = ['temp_media','temp_max','temp_min','temp_rango',
                      'temp_bulbo_humedo_rango','temp_superficie',
                      'precipitacion_mm','precipitacion_acum',
                      'humedad_media','humedad_max','humedad_min','humedad_especifica',
                      'radiacion_solar','radiacion_onda_larga',
                      'viento_medio','viento_max',
                      'presion','humedad_suelo','evapotranspiracion','punto_rocio']
    clima_vars = [v for v in clima_vars_all if v in df.columns]

    agg_dict = {v: 'sum' if 'precipitacion' in v else 'mean' for v in clima_vars}
    df_sem = df.groupby(['anio', 'semana']).agg(agg_dict).reset_index()
    return df_sem


print('fetch_nasa_power() definida. Lista para descargar.')


In [ ]:
# ── OPCIÓN A: Descargar datos reales (requiere internet, ~5-10 min) ───────────
# Descomentar para ejecutar

clima_cache = OUT_DIR / 'clima_nasa_power_2018_2025.csv'
#
if not clima_cache.exists():
    clima_dfs = []
    for prov, (lat, lon) in CENTROIDES_PROV.items():
        print(f'  Descargando {prov}...')
        try:
            d = fetch_nasa_power(lat=lat, lon=lon,
                                start_date='20180101', end_date='20251231')
            d['provincia'] = prov
            clima_dfs.append(d)
            print(f'    OK: {len(d):,} semanas')
            time.sleep(1)  # Respetar rate limit de la API
        except Exception as e:
            print(f'    ERROR: {e}')
#
    df_clima = pd.concat(clima_dfs, ignore_index=True)
    df_clima.to_csv(clima_cache, index=False)
    print(f'Guardado: {clima_cache}')
else:
        df_clima = pd.read_csv(clima_cache)
        print(f'Cargado desde caché: {len(df_clima):,} filas')

In [ ]:
# ── OPCIÓN B: Datos sintéticos calibrados (sin internet) ──────────────────────
# Misma estructura que los datos reales de NASA POWER.
# Reemplazar por Opción A en producción.

np.random.seed(42)
clima_rows = []
for anio in range(2018, 2026):
    for semana in range(1, 53):
        for prov, (lat, lon) in CENTROIDES_PROV.items():
            angle   = 2 * np.pi * (semana - 1) / 52
            t_base  = 22 - abs(lat + 32) * 0.5
            t_media = t_base + 8 * np.cos(angle + np.pi) + np.random.normal(0, 1.5)
            hum      = float(np.clip(65 + 15 * np.cos(angle + np.pi) + np.random.normal(0, 5), 30, 99))
            precip  = max(0.0, 40 * np.cos(angle + np.pi/4) + 20 + np.random.exponential(15))
            clima_rows.append({
                'anio': anio, 'semana': semana, 'provincia': prov,
                'temp_media':       round(t_media, 2),
                'temp_max':         round(t_media + 5 + np.random.normal(0, 1), 2),
                'temp_min':         round(t_media - 5 + np.random.normal(0, 1), 2),
                'temp_rango':       round(10 + np.random.exponential(2), 2),
                'humedad_media':    round(hum, 2),
                'humedad_max':      round(min(hum + 10, 99), 2),
                'humedad_min':      round(max(hum - 15, 20), 2),
                'precipitacion_mm': round(precip, 2),
                'precipitacion_acum': round(precip * 7, 2),
                'radiacion_solar':  round(max(0, 200 + 150 * np.cos(angle + np.pi) + np.random.normal(0, 30)), 2),
                'viento_medio':     round(max(0, 3 + np.random.exponential(1.5)), 2),
                'humedad_suelo':    round(float(np.clip(0.3 + 0.15 * np.cos(angle + np.pi/4) + np.random.normal(0, 0.05), 0.05, 0.9)), 3),
                'punto_rocio':      round(t_media - 5 + np.random.normal(0, 2), 2),
            })

df_clima = pd.DataFrame(clima_rows)
print(f'✅ Datos climáticos: {len(df_clima):,} filas — {df_clima.provincia.nunique()} provincias × {df_clima.anio.nunique()} años')
print('⚠️  Actualmente usando datos sintéticos. Descomentar Opción A para datos reales de NASA POWER.')

In [ ]:
# ── 8.1 Limpieza del dataset SNVS ─────────────────────────────────────────────
print('=== REPORTE DE CALIDAD — SNVS ===')
print(f'Filas originales         : {len(df_raw):,}')
print(f'Nulos en cantidad_casos  : {df_raw.cantidad_casos.isna().sum()}')
print(f'Departamentos únicos     : {df_raw.departamento.nunique()}')
print(f'Rango de años            : {df_raw.anio.min()} – {df_raw.anio.max()}')
print()

# Decisión 1: excluir registros sin departamento identificable
# Justificación: no pueden asignarse geográficamente para el modelo espacial
mask_desconocido = df_raw['departamento'].str.lower().str.strip().isin(
    ['desconocido', '*sin dato*', 'sin dato', 'nd', '']
)
n_desconocido = mask_desconocido.sum()
print(f'Filas con departamento desconocido: {n_desconocido:,} → EXCLUIDAS')

# Decisión 2: excluir años fuera del rango de interés
mask_anio = (df_raw['anio'] < 2018) | (df_raw['anio'] > 2025)
n_anio = mask_anio.sum()
print(f'Filas con año fuera de 2018–2025 : {n_anio:,} → EXCLUIDAS')

# Decisión 3: corregir semana 53 → 52 (artefacto de calendario ISO)
n_se53 = (df_raw['semana'] == 53).sum()
df_raw.loc[df_raw['semana'] == 53, 'semana'] = 52
print(f'Filas con SE=53 corregidas a 52  : {n_se53:,}')

# Aplicar filtros
df_snvs = df_raw[~mask_desconocido & ~mask_anio].copy()

# Decisión 4: los duplicados del archivo 2025 (primer y segundo semestre)
# pueden solaparse en SE 1–15. Deduplicar por (anio, semana, departamento).
n_antes = len(df_snvs)
df_snvs = df_snvs.groupby(
    ['anio', 'semana', 'departamento', 'provincia'], as_index=False
)['cantidad_casos'].sum()
n_despues = len(df_snvs)
print(f'Filas después de deduplicación   : {n_despues:,} (antes: {n_antes:,})')

# Reporte final
print(f'\n✅ Dataset SNVS limpio           : {len(df_snvs):,} filas | {df_snvs.cantidad_casos.sum():,} casos')
print(f'   Departamentos únicos          : {df_snvs.departamento.nunique()}')
print(f'   Provincias                    : {df_snvs.provincia.nunique()}')


In [ ]:
# ── 8.2 Limpieza del dataset climático ────────────────────────────────────────
print('=== REPORTE DE CALIDAD — CLIMA ===')
print(f'Filas originales   : {len(df_clima):,}')
print(f'Nulos por columna:')
nulos = df_clima.isnull().sum()
print(nulos[nulos > 0].to_string() if (nulos > 0).any() else '  Ninguno')

# Decisión: valores -999 de NASA POWER (missing) → interpolación lineal por provincia
clima_vars_num = df_clima.select_dtypes(include='number').columns.difference(['anio', 'semana'])
for var in clima_vars_num:
    n_miss = df_clima[var].isna().sum()
    if n_miss > 0:
        df_clima[var] = (df_clima.groupby('provincia')[var]
                                  .transform(lambda x: x.interpolate(method='linear', limit_direction='both')))
        print(f'  Interpolados {n_miss} valores en {var}')

print(f'\n✅ Dataset climático limpio: {len(df_clima):,} filas')


In [ ]:
# ── 8.3 Join epidemiológico × climático ───────────────────────────────────────

# Normalizar nombres de provincia para el join
PROV_MAP = {
    'Ciudad De Buenos Aires': 'CABA',
    'Ciudad de Buenos Aires': 'CABA',
    'Caba': 'CABA',
    'Entre Rios': 'Entre Ríos',
    'Cordoba': 'Córdoba',
    'Tucuman': 'Tucumán',
    'Santiago Del Estero': 'Santiago del Estero',
}
df_snvs['provincia_norm'] = (df_snvs['provincia'].str.strip()
                                                  .str.title()
                                                  .replace(PROV_MAP))

df_master = df_snvs.merge(
    df_clima.rename(columns={'provincia': 'provincia_norm'}),
    on=['anio', 'semana', 'provincia_norm'],
    how='left'
)

cobertura_clima = df_master['temp_media'].notna().mean() * 100
print(f'✅ Dataset maestro: {len(df_master):,} filas')
print(f'   Cobertura climática: {cobertura_clima:.1f}% de las filas')
print(f'   Columnas: {list(df_master.columns)}')
df_master.head(3)


In [ ]:
# ── 8.4 Reporte de calidad final ─────────────────────────────────────────────
print('=' * 55)
print('REPORTE DE CALIDAD — DATASET MAESTRO FINAL')
print('=' * 55)
print(f'Filas totales               : {len(df_master):,}')
print(f'Columnas                    : {df_master.shape[1]}')
print(f'Años cubiertos              : {sorted(df_master.anio.unique())}')
print(f'Departamentos únicos        : {df_master.departamento.nunique()}')
print(f'Provincias                  : {df_master.provincia.nunique()}')
print(f'Total casos confirmados     : {df_master.cantidad_casos.sum():,}')
print(f'Semanas epidemiológicas     : {df_master.semana.min()} – {df_master.semana.max()}')
print()
print('Estadísticas de casos por departamento-semana:')
print(df_master['cantidad_casos'].describe().to_string())
print()
print('Nulos por columna:')
nulos_final = df_master.isnull().sum()
print(nulos_final[nulos_final > 0].to_string() if (nulos_final > 0).any() else '  Ninguno')


In [ ]:
# ── 8.1 Limpieza del dataset SNVS ─────────────────────────────────────────────
print('=== REPORTE DE CALIDAD — SNVS ===')
print(f'Filas originales         : {len(df_raw):,}')
print(f'Nulos en cantidad_casos  : {df_raw.cantidad_casos.isna().sum()}')
print(f'Departamentos únicos     : {df_raw.departamento.nunique()}')
print(f'Rango de años            : {df_raw.anio.min()} – {df_raw.anio.max()}')
print()

# Decisión 1: excluir registros sin departamento identificable
# Justificación: no pueden asignarse geográficamente para el modelo espacial
mask_desconocido = df_raw['departamento'].str.lower().str.strip().isin(
    ['desconocido', '*sin dato*', 'sin dato', 'nd', '']
)
n_desconocido = mask_desconocido.sum()
print(f'Filas con departamento desconocido: {n_desconocido:,} → EXCLUIDAS')

# Decisión 2: excluir años fuera del rango de interés
mask_anio = (df_raw['anio'] < 2018) | (df_raw['anio'] > 2025)
n_anio = mask_anio.sum()
print(f'Filas con año fuera de 2018–2025 : {n_anio:,} → EXCLUIDAS')

# Decisión 3: corregir semana 53 → 52 (artefacto de calendario ISO)
n_se53 = (df_raw['semana'] == 53).sum()
df_raw.loc[df_raw['semana'] == 53, 'semana'] = 52
print(f'Filas con SE=53 corregidas a 52  : {n_se53:,}')

# Aplicar filtros
df_snvs = df_raw[~mask_desconocido & ~mask_anio].copy()

# Decisión 4: los duplicados del archivo 2025 (primer y segundo semestre)
# pueden solaparse en SE 1–15. Deduplicar por (anio, semana, departamento).
n_antes = len(df_snvs)
df_snvs = df_snvs.groupby(
    ['anio', 'semana', 'departamento', 'provincia'], as_index=False
)['cantidad_casos'].sum()
n_despues = len(df_snvs)
print(f'Filas después de deduplicación   : {n_despues:,} (antes: {n_antes:,})')

# Reporte final
print(f'\n✅ Dataset SNVS limpio           : {len(df_snvs):,} filas | {df_snvs.cantidad_casos.sum():,} casos')
print(f'   Departamentos únicos          : {df_snvs.departamento.nunique()}')
print(f'   Provincias                    : {df_snvs.provincia.nunique()}')

In [ ]:
# ── 8.2 Limpieza del dataset climático ────────────────────────────────────────
print('=== REPORTE DE CALIDAD — CLIMA ===')
print(f'Filas originales   : {len(df_clima):,}')
print(f'Nulos por columna:')
nulos = df_clima.isnull().sum()
print(nulos[nulos > 0].to_string() if (nulos > 0).any() else '  Ninguno')

# Decisión: valores -999 de NASA POWER (missing) → interpolación lineal por provincia
clima_vars_num = df_clima.select_dtypes(include='number').columns.difference(['anio', 'semana'])
for var in clima_vars_num:
    n_miss = df_clima[var].isna().sum()
    if n_miss > 0:
        df_clima[var] = (df_clima.groupby('provincia')[var]
                                  .transform(lambda x: x.interpolate(method='linear', limit_direction='both')))
        print(f'  Interpolados {n_miss} valores en {var}')

print(f'\n✅ Dataset climático limpio: {len(df_clima):,} filas')

In [ ]:
# ── 8.3 Join epidemiológico × climático ───────────────────────────────────────

# Normalizar nombres de provincia para el join
PROV_MAP = {
    'Ciudad De Buenos Aires': 'CABA',
    'Ciudad de Buenos Aires': 'CABA',
    'Caba': 'CABA',
    'Entre Rios': 'Entre Ríos',
    'Cordoba': 'Córdoba',
    'Tucuman': 'Tucumán',
    'Santiago Del Estero': 'Santiago del Estero',
}
df_snvs['provincia_norm'] = (df_snvs['provincia'].str.strip()
                                                  .str.title()
                                                  .replace(PROV_MAP))

df_master = df_snvs.merge(
    df_clima.rename(columns={'provincia': 'provincia_norm'}),
    on=['anio', 'semana', 'provincia_norm'],
    how='left'
)

cobertura_clima = df_master['temp_media'].notna().mean() * 100
print(f'✅ Dataset maestro: {len(df_master):,} filas')
print(f'   Cobertura climática: {cobertura_clima:.1f}% de las filas')
print(f'   Columnas: {list(df_master.columns)}')
df_master.head(3)

In [ ]:
# ── 8.4 Reporte de calidad final ─────────────────────────────────────────────
print('=' * 55)
print('REPORTE DE CALIDAD — DATASET MAESTRO FINAL')
print('=' * 55)
print(f'Filas totales               : {len(df_master):,}')
print(f'Columnas                    : {df_master.shape[1]}')
print(f'Años cubiertos              : {sorted(df_master.anio.unique())}')
print(f'Departamentos únicos        : {df_master.departamento.nunique()}')
print(f'Provincias                  : {df_master.provincia.nunique()}')
print(f'Total casos confirmados     : {df_master.cantidad_casos.sum():,}')
print(f'Semanas epidemiológicas     : {df_master.semana.min()} – {df_master.semana.max()}')
print()
print('Estadísticas de casos por departamento-semana:')
print(df_master['cantidad_casos'].describe().to_string())
print()
print('Nulos por columna:')
nulos_final = df_master.isnull().sum()
print(nulos_final[nulos_final > 0].to_string() if (nulos_final > 0).any() else '  Ninguno')

In [ ]:
# Exportar dataset completo
ruta_completo = OUT_DIR / 'dengue_dataset_limpio.csv'
df_master.to_csv(ruta_completo, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_completo}  ({len(df_master):,} filas)')

# Subconjunto: departamentos con suficiente historia
conteo_depto = df_master.groupby('departamento').size()
deptos_suficientes = conteo_depto[conteo_depto >= 30].index
df_modelado = df_master[df_master.departamento.isin(deptos_suficientes)].copy()

ruta_modelado = OUT_DIR / 'dengue_dataset_departamentos_top.csv'
df_modelado.to_csv(ruta_modelado, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_modelado}  ({len(df_modelado):,} filas | {df_modelado.departamento.nunique()} departamentos)')

# Diccionario de variables
diccionario = pd.DataFrame([
    {'variable': 'anio',              'tipo': 'int',   'descripcion': 'Año calendario'},
    {'variable': 'semana',            'tipo': 'int',   'descripcion': 'Semana epidemiológica ISO (1–52)'},
    {'variable': 'departamento',      'tipo': 'str',   'descripcion': 'Nombre del departamento/partido'},
    {'variable': 'provincia',         'tipo': 'str',   'descripcion': 'Nombre de la provincia'},
    {'variable': 'cantidad_casos',    'tipo': 'int',   'descripcion': 'Casos confirmados de dengue (TARGET)'},
    {'variable': 'temp_media',        'tipo': 'float', 'descripcion': 'Temperatura media 2m (°C) — NASA POWER T2M'},
    {'variable': 'temp_max',          'tipo': 'float', 'descripcion': 'Temperatura máxima 2m (°C) — NASA POWER T2M_MAX'},
    {'variable': 'temp_min',          'tipo': 'float', 'descripcion': 'Temperatura mínima 2m (°C) — NASA POWER T2M_MIN'},
    {'variable': 'temp_rango',        'tipo': 'float', 'descripcion': 'Amplitud térmica diaria (°C) — NASA POWER T2M_RANGE'},
    {'variable': 'humedad_media',     'tipo': 'float', 'descripcion': 'Humedad relativa media 2m (%) — NASA POWER RH2M'},
    {'variable': 'humedad_max',       'tipo': 'float', 'descripcion': 'Humedad relativa máxima (%)'},
    {'variable': 'humedad_min',       'tipo': 'float', 'descripcion': 'Humedad relativa mínima (%)'},
    {'variable': 'precipitacion_mm',  'tipo': 'float', 'descripcion': 'Precipitación diaria corregida (mm) — NASA POWER PRECTOTCORR'},
    {'variable': 'precipitacion_acum','tipo': 'float', 'descripcion': 'Precipitación acumulada semanal (mm)'},
    {'variable': 'radiacion_solar',   'tipo': 'float', 'descripcion': 'Radiación solar incidente (W/m²) — NASA POWER ALLSKY_SFC_SW_DWN'},
    {'variable': 'viento_medio',      'tipo': 'float', 'descripcion': 'Velocidad del viento media 2m (m/s) — NASA POWER WS2M'},
    {'variable': 'humedad_suelo',     'tipo': 'float', 'descripcion': 'Humedad del suelo capa raíz — NASA POWER GWETROOT'},
    {'variable': 'punto_rocio',       'tipo': 'float', 'descripcion': 'Temperatura de punto de rocío (°C) — NASA POWER T2MDEW'},
])

ruta_diccionario = OUT_DIR / 'diccionario_variables.csv'
diccionario.to_csv(ruta_diccionario, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_diccionario}')

In [ ]:
# Exportar dataset completo
ruta_completo = OUT_DIR / 'dengue_dataset_limpio.csv'
df_master.to_csv(ruta_completo, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_completo}  ({len(df_master):,} filas)')

# Subconjunto: departamentos con suficiente historia
conteo_depto = df_master.groupby('departamento').size()
deptos_suficientes = conteo_depto[conteo_depto >= 30].index
df_modelado = df_master[df_master.departamento.isin(deptos_suficientes)].copy()

ruta_modelado = OUT_DIR / 'dengue_dataset_departamentos_top.csv'
df_modelado.to_csv(ruta_modelado, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_modelado}  ({len(df_modelado):,} filas | {df_modelado.departamento.nunique()} departamentos)')

# Diccionario de variables
diccionario = pd.DataFrame([
    {'variable': 'anio',              'tipo': 'int',   'descripcion': 'Año calendario'},
    {'variable': 'semana',            'tipo': 'int',   'descripcion': 'Semana epidemiológica ISO (1–52)'},
    {'variable': 'departamento',      'tipo': 'str',   'descripcion': 'Nombre del departamento/partido'},
    {'variable': 'provincia',         'tipo': 'str',   'descripcion': 'Nombre de la provincia'},
    {'variable': 'cantidad_casos',    'tipo': 'int',   'descripcion': 'Casos confirmados de dengue (TARGET)'},
    {'variable': 'temp_media',        'tipo': 'float', 'descripcion': 'Temperatura media 2m (°C) — NASA POWER T2M'},
    {'variable': 'temp_max',          'tipo': 'float', 'descripcion': 'Temperatura máxima 2m (°C) — NASA POWER T2M_MAX'},
    {'variable': 'temp_min',          'tipo': 'float', 'descripcion': 'Temperatura mínima 2m (°C) — NASA POWER T2M_MIN'},
    {'variable': 'temp_rango',        'tipo': 'float', 'descripcion': 'Amplitud térmica diaria (°C) — NASA POWER T2M_RANGE'},
    {'variable': 'humedad_media',     'tipo': 'float', 'descripcion': 'Humedad relativa media 2m (%) — NASA POWER RH2M'},
    {'variable': 'humedad_max',       'tipo': 'float', 'descripcion': 'Humedad relativa máxima (%)'},
    {'variable': 'humedad_min',       'tipo': 'float', 'descripcion': 'Humedad relativa mínima (%)'},
    {'variable': 'precipitacion_mm',  'tipo': 'float', 'descripcion': 'Precipitación diaria corregida (mm) — NASA POWER PRECTOTCORR'},
    {'variable': 'precipitacion_acum','tipo': 'float', 'descripcion': 'Precipitación acumulada semanal (mm)'},
    {'variable': 'radiacion_solar',   'tipo': 'float', 'descripcion': 'Radiación solar incidente (W/m²) — NASA POWER ALLSKY_SFC_SW_DWN'},
    {'variable': 'viento_medio',      'tipo': 'float', 'descripcion': 'Velocidad del viento media 2m (m/s) — NASA POWER WS2M'},
    {'variable': 'humedad_suelo',     'tipo': 'float', 'descripcion': 'Humedad del suelo capa raíz — NASA POWER GWETROOT'},
    {'variable': 'punto_rocio',       'tipo': 'float', 'descripcion': 'Temperatura de punto de rocío (°C) — NASA POWER T2MDEW'},
])

ruta_diccionario = OUT_DIR / 'diccionario_variables.csv'
diccionario.to_csv(ruta_diccionario, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_diccionario}')

In [ ]:
# ── 8.1 Limpieza del dataset SNVS ─────────────────────────────────────────────
print('=== REPORTE DE CALIDAD — SNVS ===')
print(f'Filas originales         : {len(df_raw):,}')
print(f'Nulos en cantidad_casos  : {df_raw.cantidad_casos.isna().sum()}')
print(f'Departamentos únicos     : {df_raw.departamento.nunique()}')
print(f'Rango de años            : {df_raw.anio.min()} – {df_raw.anio.max()}')
print()

# Decisión 1: excluir registros sin departamento identificable
# Justificación: no pueden asignarse geográficamente para el modelo espacial
mask_desconocido = df_raw['departamento'].str.lower().str.strip().isin(
    ['desconocido', '*sin dato*', 'sin dato', 'nd', '']
)
n_desconocido = mask_desconocido.sum()
print(f'Filas con departamento desconocido: {n_desconocido:,} → EXCLUIDAS')

# Decisión 2: excluir años fuera del rango de interés
mask_anio = (df_raw['anio'] < 2018) | (df_raw['anio'] > 2025)
n_anio = mask_anio.sum()
print(f'Filas con año fuera de 2018–2025 : {n_anio:,} → EXCLUIDAS')

# Decisión 3: corregir semana 53 → 52 (artefacto de calendario ISO)
n_se53 = (df_raw['semana'] == 53).sum()
df_raw.loc[df_raw['semana'] == 53, 'semana'] = 52
print(f'Filas con SE=53 corregidas a 52  : {n_se53:,}')

# Aplicar filtros
df_snvs = df_raw[~mask_desconocido & ~mask_anio].copy()

# Decisión 4: los duplicados del archivo 2025 (primer y segundo semestre)
# pueden solaparse en SE 1–15. Deduplicar por (anio, semana, departamento).
n_antes = len(df_snvs)
df_snvs = df_snvs.groupby(
    ['anio', 'semana', 'departamento', 'provincia'], as_index=False
)['cantidad_casos'].sum()
n_despues = len(df_snvs)
print(f'Filas después de deduplicación   : {n_despues:,} (antes: {n_antes:,})')

# Reporte final
print(f'\n✅ Dataset SNVS limpio           : {len(df_snvs):,} filas | {df_snvs.cantidad_casos.sum():,} casos')
print(f'   Departamentos únicos          : {df_snvs.departamento.nunique()}')
print(f'   Provincias                    : {df_snvs.provincia.nunique()}')


In [ ]:
# ── 8.2 Limpieza del dataset climático ────────────────────────────────────────
print('=== REPORTE DE CALIDAD — CLIMA ===')
print(f'Filas originales   : {len(df_clima):,}')
print(f'Nulos por columna:')
nulos = df_clima.isnull().sum()
print(nulos[nulos > 0].to_string() if (nulos > 0).any() else '  Ninguno')

# Decisión: valores -999 de NASA POWER (missing) → interpolación lineal por provincia
clima_vars_num = df_clima.select_dtypes(include='number').columns.difference(['anio', 'semana'])
for var in clima_vars_num:
    n_miss = df_clima[var].isna().sum()
    if n_miss > 0:
        df_clima[var] = (df_clima.groupby('provincia')[var]
                                  .transform(lambda x: x.interpolate(method='linear', limit_direction='both')))
        print(f'  Interpolados {n_miss} valores en {var}')

print(f'\n✅ Dataset climático limpio: {len(df_clima):,} filas')

In [ ]:
# ── 8.3 Join epidemiológico × climático ───────────────────────────────────────

# Normalizar nombres de provincia para el join
PROV_MAP = {
    'Ciudad De Buenos Aires': 'CABA',
    'Ciudad de Buenos Aires': 'CABA',
    'Caba': 'CABA',
    'Entre Rios': 'Entre Ríos',
    'Cordoba': 'Córdoba',
    'Tucuman': 'Tucumán',
    'Santiago Del Estero': 'Santiago del Estero',
}
df_snvs['provincia_norm'] = (df_snvs['provincia'].str.strip()
                                                  .str.title()
                                                  .replace(PROV_MAP))

df_master = df_snvs.merge(
    df_clima.rename(columns={'provincia': 'provincia_norm'}),
    on=['anio', 'semana', 'provincia_norm'],
    how='left'
)

cobertura_clima = df_master['temp_media'].notna().mean() * 100
print(f'✅ Dataset maestro: {len(df_master):,} filas')
print(f'   Cobertura climática: {cobertura_clima:.1f}% de las filas')
print(f'   Columnas: {list(df_master.columns)}')
df_master.head(3)

In [ ]:
# ── 8.4 Reporte de calidad final ─────────────────────────────────────────────
print('=' * 55)
print('REPORTE DE CALIDAD — DATASET MAESTRO FINAL')
print('=' * 55)
print(f'Filas totales               : {len(df_master):,}')
print(f'Columnas                    : {df_master.shape[1]}')
print(f'Años cubiertos              : {sorted(df_master.anio.unique())}')
print(f'Departamentos únicos        : {df_master.departamento.nunique()}')
print(f'Provincias                  : {df_master.provincia.nunique()}')
print(f'Total casos confirmados     : {df_master.cantidad_casos.sum():,}')
print(f'Semanas epidemiológicas     : {df_master.semana.min()} – {df_master.semana.max()}')
print()
print('Estadísticas de casos por departamento-semana:')
print(df_master['cantidad_casos'].describe().to_string())
print()
print('Nulos por columna:')
nulos_final = df_master.isnull().sum()
print(nulos_final[nulos_final > 0].to_string() if (nulos_final > 0).any() else '  Ninguno')

In [ ]:
# Exportar dataset completo
ruta_completo = OUT_DIR / 'dengue_dataset_limpio.csv'
df_master.to_csv(ruta_completo, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_completo}  ({len(df_master):,} filas)')

# Subconjunto: departamentos con suficiente historia
conteo_depto = df_master.groupby('departamento').size()
deptos_suficientes = conteo_depto[conteo_depto >= 30].index
df_modelado = df_master[df_master.departamento.isin(deptos_suficientes)].copy()

ruta_modelado = OUT_DIR / 'dengue_dataset_departamentos_top.csv'
df_modelado.to_csv(ruta_modelado, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_modelado}  ({len(df_modelado):,} filas | {df_modelado.departamento.nunique()} departamentos)')

# Diccionario de variables
diccionario = pd.DataFrame([
    {'variable': 'anio',              'tipo': 'int',   'descripcion': 'Año calendario'},
    {'variable': 'semana',            'tipo': 'int',   'descripcion': 'Semana epidemiológica ISO (1–52)'},
    {'variable': 'departamento',      'tipo': 'str',   'descripcion': 'Nombre del departamento/partido'},
    {'variable': 'provincia',         'tipo': 'str',   'descripcion': 'Nombre de la provincia'},
    {'variable': 'cantidad_casos',    'tipo': 'int',   'descripcion': 'Casos confirmados de dengue (TARGET)'},
    {'variable': 'temp_media',        'tipo': 'float', 'descripcion': 'Temperatura media 2m (°C) — NASA POWER T2M'},
    {'variable': 'temp_max',          'tipo': 'float', 'descripcion': 'Temperatura máxima 2m (°C) — NASA POWER T2M_MAX'},
    {'variable': 'temp_min',          'tipo': 'float', 'descripcion': 'Temperatura mínima 2m (°C) — NASA POWER T2M_MIN'},
    {'variable': 'temp_rango',        'tipo': 'float', 'descripcion': 'Amplitud térmica diaria (°C) — NASA POWER T2M_RANGE'},
    {'variable': 'humedad_media',     'tipo': 'float', 'descripcion': 'Humedad relativa media 2m (%) — NASA POWER RH2M'},
    {'variable': 'humedad_max',       'tipo': 'float', 'descripcion': 'Humedad relativa máxima (%)'},
    {'variable': 'humedad_min',       'tipo': 'float', 'descripcion': 'Humedad relativa mínima (%)'},
    {'variable': 'precipitacion_mm',  'tipo': 'float', 'descripcion': 'Precipitación diaria corregida (mm) — NASA POWER PRECTOTCORR'},
    {'variable': 'precipitacion_acum','tipo': 'float', 'descripcion': 'Precipitación acumulada semanal (mm)'},
    {'variable': 'radiacion_solar',   'tipo': 'float', 'descripcion': 'Radiación solar incidente (W/m²) — NASA POWER ALLSKY_SFC_SW_DWN'},
    {'variable': 'viento_medio',      'tipo': 'float', 'descripcion': 'Velocidad del viento media 2m (m/s) — NASA POWER WS2M'},
    {'variable': 'humedad_suelo',     'tipo': 'float', 'descripcion': 'Humedad del suelo capa raíz — NASA POWER GWETROOT'},
    {'variable': 'punto_rocio',       'tipo': 'float', 'descripcion': 'Temperatura de punto de rocío (°C) — NASA POWER T2MDEW'},
])

ruta_diccionario = OUT_DIR / 'diccionario_variables.csv'
diccionario.to_csv(ruta_diccionario, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_diccionario}')

In [ ]:
# ── Parámetros seleccionados (incluyendo MO/DY para robustez) ─────────────
NASA_PARAMS = ','.join([
    # Temperatura (3)
    'T2M', 'T2M_MAX', 'T2M_MIN',
    # Agua y humedad (2)
    'PRECTOTCORR', 'RH2M',
    # Viento (1)
    'WS2M',
    # Fecha (2) - para robustez en construccion de fecha
    'MO', 'DY'
])

# Centroides por provincia (lat, lon)
CENTROIDES_PROV = {
    'Buenos Aires':        (-36.60, -60.00),
    'CABA':                (-34.61, -58.44),
    'Córdoba':             (-31.42, -64.18),
    'Santa Fe':            (-31.63, -60.70),
    'Tucumán':             (-26.82, -65.22),
    'Salta':               (-24.79, -65.41),
    'Misiones':            (-27.36, -55.90),
    'Jujuy':               (-24.18, -65.30),
    'Entre Ríos':          (-31.74, -60.54),
    'Chaco':               (-27.46, -58.99),
    'Formosa':             (-26.18, -58.18),
    'Corrientes':          (-27.47, -58.83),
    'Santiago del Estero': (-27.78, -64.26),
}

print(f'Parámetros a descargar: {len(NASA_PARAMS.split(","))} variables')
print(f'Provincias: {len(CENTROIDES_PROV)}')


In [ ]:
def fetch_nasa_power(lat: float, lon: float,
                     start_date: str = '20180101',
                     end_date:   str = '20251231',
                     retries: int = 3) -> pd.DataFrame:
    """
    Descarga datos climáticos DIARIOS de NASA POWER para una coordenada.
    Formato CSV — evita pivoteo de JSON. Sin API key.

    Parámetros
    ----------
    lat, lon    : coordenadas del centroide
    start_date  : fecha inicio YYYYMMDD
    end_date    : fecha fin   YYYYMMDD
    retries     : intentos ante errores de red

    Retorna
    -------
    DataFrame con columnas: fecha, anio, semana + variables climáticas
    """
    url = (
        f'https://power.larc.nasa.gov/api/temporal/daily/point'
        f'?parameters={NASA_PARAMS}'
        f'&community=SB'
        f'&longitude={lon}'
        f'&latitude={lat}'
        f'&start={start_date}'
        f'&end={end_date}'
        f'&format=CSV'
    )

    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=120)
            resp.raise_for_status()
            break
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f'    Reintento {attempt + 1}/{retries}...')
            time.sleep(5)

    # La API agrega cabeceras de metadatos antes del CSV real.
    # Saltamos hasta encontrar la línea con los nombres de columna (empieza con 'YEAR').
    lines = resp.text.splitlines()
    try:
        header_idx = next(i for i, line in enumerate(lines) if line.startswith('YEAR'))
    except StopIteration:
        raise ValueError('Respuesta inesperada de NASA POWER — no se encontró encabezado CSV')

    df = pd.read_csv(
        StringIO('\n'.join(lines[header_idx:])),
        na_values='-999',
    )

    # Construir fecha y semana epidemiológica de forma robusta
    if 'YEAR' in df.columns and 'DOY' in df.columns:
        df['fecha'] = pd.to_datetime(
            df['YEAR'].astype(str) + df['DOY'].astype(str).str.zfill(3),
            format='%Y%j'
        )
    elif 'YEAR' in df.columns and 'MO' in df.columns and 'DY' in df.columns:
        df['fecha'] = pd.to_datetime(
            df['YEAR'].astype(str) + '-' + df['MO'].astype(str) + '-' + df['DY'].astype(str),
            format='%Y-%m-%d'
        )
    else:
        raise ValueError(
            "NASA POWER response missing sufficient date columns (neither DOY nor MO/DY found).")

    df['anio']   = df['fecha'].dt.isocalendar().year.astype(int)
    df['semana'] = df['fecha'].dt.isocalendar().week.astype(int)

    # Renombrar a nombres canónicos (se han simplificado algunos, mantener los que existen)
    df = df.rename(columns={
        'T2M':              'temp_media',
        'T2M_MAX':          'temp_max',
        'T2M_MIN':          'temp_min',
        'PRECTOTCORR':      'precipitacion_mm',
        'RH2M':             'humedad_media',
        'WS2M':             'viento_medio',
        'T2M_RANGE':        'temp_rango',
        'T_WET_RANGE_2M':   'temp_bulbo_humedo_rango',
        'TS':               'temp_superficie',
        'PRECTOTCORR_SUM':  'precipitacion_acum',
        'RH2M_MAX':         'humedad_max',
        'RH2M_MIN':         'humedad_min',
        'QV2M':             'humedad_especifica',
        'ALLSKY_SFC_SW_DWN':'radiacion_solar',
        'ALLSKY_SFC_LW_DWN':'radiacion_onda_larga',
        'WS2M_MAX':         'viento_max',
        'PS':               'presion',
        'GWETROOT':         'humedad_suelo',
        'EVPTRNS':          'evapotranspiracion',
        'T2MDEW':           'punto_rocio',
    })

    # Agregar de diario a semanal (media; precipitación como suma)
    # Asegurarse de que solo se usen las columnas presentes en el DataFrame
    clima_vars_all = ['temp_media','temp_max','temp_min','temp_rango',
                      'temp_bulbo_humedo_rango','temp_superficie',
                      'precipitacion_mm','precipitacion_acum',
                      'humedad_media','humedad_max','humedad_min','humedad_especifica',
                      'radiacion_solar','radiacion_onda_larga',
                      'viento_medio','viento_max',
                      'presion','humedad_suelo','evapotranspiracion','punto_rocio']
    clima_vars = [v for v in clima_vars_all if v in df.columns]

    agg_dict = {v: 'sum' if 'precipitacion' in v else 'mean' for v in clima_vars}
    df_sem = df.groupby(['anio', 'semana']).agg(agg_dict).reset_index()
    return df_sem


print('fetch_nasa_power() definida. Lista para descargar.')


In [ ]:
# ── OPCIÓN A: Descargar datos reales (requiere internet, ~5-10 min) ───────────
# Descomentar para ejecutar

clima_cache = OUT_DIR / 'clima_nasa_power_2018_2025.csv'
#
if not clima_cache.exists():
    clima_dfs = []
    for prov, (lat, lon) in CENTROIDES_PROV.items():
        print(f'  Descargando {prov}...')
        try:
            d = fetch_nasa_power(lat=lat, lon=lon,
                                start_date='20180101', end_date='20251231')
            d['provincia'] = prov
            clima_dfs.append(d)
            print(f'    OK: {len(d):,} semanas')
            time.sleep(1)  # Respetar rate limit de la API
        except Exception as e:
            print(f'    ERROR: {e}')
#
    df_clima = pd.concat(clima_dfs, ignore_index=True)
    df_clima.to_csv(clima_cache, index=False)
    print(f'Guardado: {clima_cache}')
else:
        df_clima = pd.read_csv(clima_cache)
        print(f'Cargado desde caché: {len(df_clima):,} filas')

In [ ]:
# ── Parámetros seleccionados (incluyendo MO/DY para robustez) ─────────────
NASA_PARAMS = ','.join([
    # Temperatura (3)
    'T2M', 'T2M_MAX', 'T2M_MIN',
    # Agua y humedad (2)
    'PRECTOTCORR', 'RH2M',
    # Viento (1)
    'WS2M',
    # Fecha (2) - para robustez en construccion de fecha
    'MO', 'DY'
])

# Centroides por provincia (lat, lon)
CENTROIDES_PROV = {
    'Buenos Aires':        (-36.60, -60.00),
    'CABA':                (-34.61, -58.44),
    'Córdoba':             (-31.42, -64.18),
    'Santa Fe':            (-31.63, -60.70),
    'Tucumán':             (-26.82, -65.22),
    'Salta':               (-24.79, -65.41),
    'Misiones':            (-27.36, -55.90),
    'Jujuy':               (-24.18, -65.30),
    'Entre Ríos':          (-31.74, -60.54),
    'Chaco':               (-27.46, -58.99),
    'Formosa':             (-26.18, -58.18),
    'Corrientes':          (-27.47, -58.83),
    'Santiago del Estero': (-27.78, -64.26),
}

print(f'Parámetros a descargar: {len(NASA_PARAMS.split(","))} variables')
print(f'Provincias: {len(CENTROIDES_PROV)}')


In [ ]:
def fetch_nasa_power(lat: float, lon: float,
                     start_date: str = '20180101',
                     end_date:   str = '20251231',
                     retries: int = 3) -> pd.DataFrame:
    """
    Descarga datos climáticos DIARIOS de NASA POWER para una coordenada.
    Formato CSV — evita pivoteo de JSON. Sin API key.

    Parámetros
    ----------
    lat, lon    : coordenadas del centroide
    start_date  : fecha inicio YYYYMMDD
    end_date    : fecha fin   YYYYMMDD
    retries     : intentos ante errores de red

    Retorna
    -------
    DataFrame con columnas: fecha, anio, semana + variables climáticas
    """
    url = (
        f'https://power.larc.nasa.gov/api/temporal/daily/point'
        f'?parameters={NASA_PARAMS}'
        f'&community=SB'
        f'&longitude={lon}'
        f'&latitude={lat}'
        f'&start={start_date}'
        f'&end={end_date}'
        f'&format=CSV'
    )

    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=120)
            resp.raise_for_status()
            break
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f'    Reintento {attempt + 1}/{retries}...')
            time.sleep(5)

    # La API agrega cabeceras de metadatos antes del CSV real.
    # Saltamos hasta encontrar la línea con los nombres de columna (empieza con 'YEAR').
    lines = resp.text.splitlines()
    try:
        header_idx = next(i for i, line in enumerate(lines) if line.startswith('YEAR'))
    except StopIteration:
        raise ValueError('Respuesta inesperada de NASA POWER — no se encontró encabezado CSV')

    df = pd.read_csv(
        StringIO('\n'.join(lines[header_idx:])),
        na_values='-999',
    )

    # Construir fecha y semana epidemiológica de forma robusta
    if 'YEAR' in df.columns and 'DOY' in df.columns:
        df['fecha'] = pd.to_datetime(
            df['YEAR'].astype(str) + df['DOY'].astype(str).str.zfill(3),
            format='%Y%j'
        )
    elif 'YEAR' in df.columns and 'MO' in df.columns and 'DY' in df.columns:
        df['fecha'] = pd.to_datetime(
            df['YEAR'].astype(str) + '-' + df['MO'].astype(str) + '-' + df['DY'].astype(str),
            format='%Y-%m-%d'
        )
    else:
        raise ValueError(
            "NASA POWER response missing sufficient date columns (neither DOY nor MO/DY found).")

    df['anio']   = df['fecha'].dt.isocalendar().year.astype(int)
    df['semana'] = df['fecha'].dt.isocalendar().week.astype(int)

    # Renombrar a nombres canónicos (se han simplificado algunos, mantener los que existen)
    df = df.rename(columns={
        'T2M':              'temp_media',
        'T2M_MAX':          'temp_max',
        'T2M_MIN':          'temp_min',
        'PRECTOTCORR':      'precipitacion_mm',
        'RH2M':             'humedad_media',
        'WS2M':             'viento_medio',
        'T2M_RANGE':        'temp_rango',
        'T_WET_RANGE_2M':   'temp_bulbo_humedo_rango',
        'TS':               'temp_superficie',
        'PRECTOTCORR_SUM':  'precipitacion_acum',
        'RH2M_MAX':         'humedad_max',
        'RH2M_MIN':         'humedad_min',
        'QV2M':             'humedad_especifica',
        'ALLSKY_SFC_SW_DWN':'radiacion_solar',
        'ALLSKY_SFC_LW_DWN':'radiacion_onda_larga',
        'WS2M_MAX':         'viento_max',
        'PS':               'presion',
        'GWETROOT':         'humedad_suelo',
        'EVPTRNS':          'evapotranspiracion',
        'T2MDEW':           'punto_rocio',
    })

    # Agregar de diario a semanal (media; precipitación como suma)
    # Asegurarse de que solo se usen las columnas presentes en el DataFrame
    clima_vars_all = ['temp_media','temp_max','temp_min','temp_rango',
                      'temp_bulbo_humedo_rango','temp_superficie',
                      'precipitacion_mm','precipitacion_acum',
                      'humedad_media','humedad_max','humedad_min','humedad_especifica',
                      'radiacion_solar','radiacion_onda_larga',
                      'viento_medio','viento_max',
                      'presion','humedad_suelo','evapotranspiracion','punto_rocio']
    clima_vars = [v for v in clima_vars_all if v in df.columns]

    agg_dict = {v: 'sum' if 'precipitacion' in v else 'mean' for v in clima_vars}
    df_sem = df.groupby(['anio', 'semana']).agg(agg_dict).reset_index()
    return df_sem


print('fetch_nasa_power() definida. Lista para descargar.')


In [ ]:
# ── OPCIÓN A: Descargar datos reales (requiere internet, ~5-10 min) ───────────
# Descomentar para ejecutar

clima_cache = OUT_DIR / 'clima_nasa_power_2018_2025.csv'
#
if not clima_cache.exists():
    clima_dfs = []
    for prov, (lat, lon) in CENTROIDES_PROV.items():
        print(f'  Descargando {prov}...')
        try:
            d = fetch_nasa_power(lat=lat, lon=lon,
                                start_date='20180101', end_date='20251231')
            d['provincia'] = prov
            clima_dfs.append(d)
            print(f'    OK: {len(d):,} semanas')
            time.sleep(1)  # Respetar rate limit de la API
        except Exception as e:
            print(f'    ERROR: {e}')
#
    df_clima = pd.concat(clima_dfs, ignore_index=True)
    df_clima.to_csv(clima_cache, index=False)
    print(f'Guardado: {clima_cache}')
else:
        df_clima = pd.read_csv(clima_cache)
        print(f'Cargado desde caché: {len(df_clima):,} filas')

In [ ]:
def fetch_nasa_power(lat: float, lon: float,
                     start_date: str = '20180101',
                     end_date:   str = '20251231',
                     retries: int = 3) -> pd.DataFrame:
    """
    Descarga datos climáticos DIARIOS de NASA POWER para una coordenada.
    Formato CSV — evita pivoteo de JSON. Sin API key.

    Parámetros
    ----------
    lat, lon    : coordenadas del centroide
    start_date  : fecha inicio YYYYMMDD
    end_date    : fecha fin   YYYYMMDD
    retries     : intentos ante errores de red

    Retorna
    -------
    DataFrame con columnas: fecha, anio, semana + 20 variables climáticas
    """
    url = (
        f'https://power.larc.nasa.gov/api/temporal/daily/point'
        f'?parameters={NASA_PARAMS}'
        f'&community=SB'
        f'&longitude={lon}'
        f'&latitude={lat}'
        f'&start={start_date}'
        f'&end={end_date}'
        f'&format=CSV'
    )

    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=120)
            resp.raise_for_status()
            break
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f'    Reintento {attempt + 1}/{retries}...')
            time.sleep(5)

    # La API agrega cabeceras de metadatos antes del CSV real.
    # Saltamos hasta encontrar la línea con los nombres de columna (empieza con 'YEAR').
    lines = resp.text.splitlines()
    try:
        header_idx = next(i for i, line in enumerate(lines) if line.startswith('YEAR'))
    except StopIteration:
        raise ValueError('Respuesta inesperada de NASA POWER — no se encontró encabezado CSV')

    df = pd.read_csv(
        StringIO('\n'.join(lines[header_idx:])),
        na_values='-999',
    )

    # Construir fecha y semana epidemiológica
    df['fecha'] = pd.to_datetime(
        df['YEAR'].astype(str) + df['DOY'].astype(str).str.zfill(3),
        format='%Y%j'
    )
    df['anio']   = df['fecha'].dt.isocalendar().year.astype(int)
    df['semana'] = df['fecha'].dt.isocalendar().week.astype(int)

    # Renombrar a nombres canónicos
    df = df.rename(columns={
        'T2M':              'temp_media',
        'T2M_MAX':          'temp_max',
        'T2M_MIN':          'temp_min',
        'T2M_RANGE':        'temp_rango',
        'T_WET_RANGE_2M':   'temp_bulbo_humedo_rango',
        'TS':               'temp_superficie',
        'PRECTOTCORR':      'precipitacion_mm',
        'PRECTOTCORR_SUM':  'precipitacion_acum',
        'RH2M':             'humedad_media',
        'RH2M_MAX':         'humedad_max',
        'RH2M_MIN':         'humedad_min',
        'QV2M':             'humedad_especifica',
        'ALLSKY_SFC_SW_DWN':'radiacion_solar',
        'ALLSKY_SFC_LW_DWN':'radiacion_onda_larga',
        'WS2M':             'viento_medio',
        'WS2M_MAX':         'viento_max',
        'PS':               'presion',
        'GWETROOT':         'humedad_suelo',
        'EVPTRNS':          'evapotranspiracion',
        'T2MDEW':           'punto_rocio',
    })

    # Agregar de diario a semanal (media; precipitación como suma)
    clima_vars = ['temp_media','temp_max','temp_min','temp_rango',
                  'temp_bulbo_humedo_rango','temp_superficie',
                  'precipitacion_mm','precipitacion_acum',
                  'humedad_media','humedad_max','humedad_min','humedad_especifica',
                  'radiacion_solar','radiacion_onda_larga',
                  'viento_medio','viento_max',
                  'presion','humedad_suelo','evapotranspiracion','punto_rocio']
    clima_vars = [v for v in clima_vars if v in df.columns]

    agg_dict = {v: 'sum' if 'precipitacion' in v else 'mean' for v in clima_vars}
    df_sem = df.groupby(['anio', 'semana']).agg(agg_dict).reset_index()
    return df_sem


print('fetch_nasa_power() definida. Lista para descargar.')

In [ ]:
# ── OPCIÓN A: Descargar datos reales (requiere internet, ~5-10 min) ───────────
# Descomentar para ejecutar

clima_cache = OUT_DIR / 'clima_nasa_power_2018_2025.csv'
#
if not clima_cache.exists():
    clima_dfs = []
    for prov, (lat, lon) in CENTROIDES_PROV.items():
        print(f'  Descargando {prov}...')
        try:
            d = fetch_nasa_power(lat=lat, lon=lon,
                                start_date='20180101', end_date='20251231')
            d['provincia'] = prov
            clima_dfs.append(d)
            print(f'    OK: {len(d):,} semanas')
            time.sleep(1)  # Respetar rate limit de la API
        except Exception as e:
            print(f'    ERROR: {e}')
#
    df_clima = pd.concat(clima_dfs, ignore_index=True)
    df_clima.to_csv(clima_cache, index=False)
    print(f'Guardado: {clima_cache}')
else:
        df_clima = pd.read_csv(clima_cache)
        print(f'Cargado desde caché: {len(df_clima):,} filas')

In [ ]:
def fetch_nasa_power(lat: float, lon: float,
                     start_date: str = '20180101',
                     end_date:   str = '20251231',
                     retries: int = 3) -> pd.DataFrame:
    """
    Descarga datos climáticos DIARIOS de NASA POWER para una coordenada.
    Formato CSV — evita pivoteo de JSON. Sin API key.

    Parámetros
    ----------
    lat, lon    : coordenadas del centroide
    start_date  : fecha inicio YYYYMMDD
    end_date    : fecha fin   YYYYMMDD
    retries     : intentos ante errores de red

    Retorna
    -------
    DataFrame con columnas: fecha, anio, semana + variables climáticas
    """
    url = (
        f'https://power.larc.nasa.gov/api/temporal/daily/point'
        f'?parameters={NASA_PARAMS}'
        f'&community=SB'
        f'&longitude={lon}'
        f'&latitude={lat}'
        f'&start={start_date}'
        f'&end={end_date}'
        f'&format=CSV'
    )

    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=120)
            resp.raise_for_status()
            break
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f'    Reintento {attempt + 1}/{retries}...')
            time.sleep(5)

    # La API agrega cabeceras de metadatos antes del CSV real.
    # Saltamos hasta encontrar la línea con los nombres de columna (empieza con 'YEAR').
    lines = resp.text.splitlines()
    try:
        header_idx = next(i for i, line in enumerate(lines) if line.startswith('YEAR'))
    except StopIteration:
        raise ValueError('Respuesta inesperada de NASA POWER — no se encontró encabezado CSV')

    df = pd.read_csv(
        StringIO('\n'.join(lines[header_idx:])),
        na_values='-999',
    )

    # Construir fecha y semana epidemiológica de forma robusta
    if 'YEAR' in df.columns and 'DOY' in df.columns:
        df['fecha'] = pd.to_datetime(
            df['YEAR'].astype(str) + df['DOY'].astype(str).str.zfill(3),
            format='%Y%j'
        )
    elif 'YEAR' in df.columns and 'MO' in df.columns and 'DY' in df.columns:
        df['fecha'] = pd.to_datetime(
            df['YEAR'].astype(str) + '-' + df['MO'].astype(str) + '-' + df['DY'].astype(str),
            format='%Y-%m-%d'
        )
    else:
        raise ValueError(
            "NASA POWER response missing sufficient date columns (neither DOY nor MO/DY found).")

    df['anio']   = df['fecha'].dt.isocalendar().year.astype(int)
    df['semana'] = df['fecha'].dt.isocalendar().week.astype(int)

    # Renombrar a nombres canónicos (se han simplificado algunos, mantener los que existen)
    df = df.rename(columns={
        'T2M':              'temp_media',
        'T2M_MAX':          'temp_max',
        'T2M_MIN':          'temp_min',
        'PRECTOTCORR':      'precipitacion_mm',
        'RH2M':             'humedad_media',
        'WS2M':             'viento_medio',
        'T2M_RANGE':        'temp_rango',
        'T_WET_RANGE_2M':   'temp_bulbo_humedo_rango',
        'TS':               'temp_superficie',
        'PRECTOTCORR_SUM':  'precipitacion_acum',
        'RH2M_MAX':         'humedad_max',
        'RH2M_MIN':         'humedad_min',
        'QV2M':             'humedad_especifica',
        'ALLSKY_SFC_SW_DWN':'radiacion_solar',
        'ALLSKY_SFC_LW_DWN':'radiacion_onda_larga',
        'WS2M_MAX':         'viento_max',
        'PS':               'presion',
        'GWETROOT':         'humedad_suelo',
        'EVPTRNS':          'evapotranspiracion',
        'T2MDEW':           'punto_rocio',
    })

    # Agregar de diario a semanal (media; precipitación como suma)
    # Asegurarse de que solo se usen las columnas presentes en el DataFrame
    clima_vars_all = ['temp_media','temp_max','temp_min','temp_rango',
                      'temp_bulbo_humedo_rango','temp_superficie',
                      'precipitacion_mm','precipitacion_acum',
                      'humedad_media','humedad_max','humedad_min','humedad_especifica',
                      'radiacion_solar','radiacion_onda_larga',
                      'viento_medio','viento_max',
                      'presion','humedad_suelo','evapotranspiracion','punto_rocio']
    clima_vars = [v for v in clima_vars_all if v in df.columns]

    agg_dict = {v: 'sum' if 'precipitacion' in v else 'mean' for v in clima_vars}
    df_sem = df.groupby(['anio', 'semana']).agg(agg_dict).reset_index()
    return df_sem


print('fetch_nasa_power() definida. Lista para descargar.')


In [ ]:
# ── OPCIÓN A: Descargar datos reales (requiere internet, ~5-10 min) ───────────
# Descomentar para ejecutar

clima_cache = OUT_DIR / 'clima_nasa_power_2018_2025.csv'
#
if not clima_cache.exists():
    clima_dfs = []
    for prov, (lat, lon) in CENTROIDES_PROV.items():
        print(f'  Descargando {prov}...')
        try:
            d = fetch_nasa_power(lat=lat, lon=lon,
                                start_date='20180101', end_date='20251231')
            d['provincia'] = prov
            clima_dfs.append(d)
            print(f'    OK: {len(d):,} semanas')
            time.sleep(1)  # Respetar rate limit de la API
        except Exception as e:
            print(f'    ERROR: {e}')
#
    df_clima = pd.concat(clima_dfs, ignore_index=True)
    df_clima.to_csv(clima_cache, index=False)
    print(f'Guardado: {clima_cache}')
else:
        df_clima = pd.read_csv(clima_cache)
        print(f'Cargado desde caché: {len(df_clima):,} filas')


In [ ]:
# ── OPCIÓN B: Datos sintéticos calibrados (sin internet) ──────────────────────
# Misma estructura que los datos reales de NASA POWER.
# Reemplazar por Opción A en producción.

np.random.seed(42)
clima_rows = []
for anio in range(2018, 2026):
    for semana in range(1, 53):
        for prov, (lat, lon) in CENTROIDES_PROV.items():
            angle   = 2 * np.pi * (semana - 1) / 52
            t_base  = 22 - abs(lat + 32) * 0.5
            t_media = t_base + 8 * np.cos(angle + np.pi) + np.random.normal(0, 1.5)
            hum     = float(np.clip(65 + 15 * np.cos(angle + np.pi) + np.random.normal(0, 5), 30, 99))
            precip  = max(0.0, 40 * np.cos(angle + np.pi/4) + 20 + np.random.exponential(15))
            clima_rows.append({
                'anio': anio, 'semana': semana, 'provincia': prov,
                'temp_media':       round(t_media, 2),
                'temp_max':         round(t_media + 5 + np.random.normal(0, 1), 2),
                'temp_min':         round(t_media - 5 + np.random.normal(0, 1), 2),
                'temp_rango':       round(10 + np.random.exponential(2), 2),
                'humedad_media':    round(hum, 2),
                'humedad_max':      round(min(hum + 10, 99), 2),
                'humedad_min':      round(max(hum - 15, 20), 2),
                'precipitacion_mm': round(precip, 2),
                'precipitacion_acum': round(precip * 7, 2),
                'radiacion_solar':  round(max(0, 200 + 150 * np.cos(angle + np.pi) + np.random.normal(0, 30)), 2),
                'viento_medio':     round(max(0, 3 + np.random.exponential(1.5)), 2),
                'humedad_suelo':    round(float(np.clip(0.3 + 0.15 * np.cos(angle + np.pi/4) + np.random.normal(0, 0.05), 0.05, 0.9)), 3),
                'punto_rocio':      round(t_media - 5 + np.random.normal(0, 2), 2),
            })

df_clima = pd.DataFrame(clima_rows)
print(f'✅ Datos climáticos: {len(df_clima):,} filas — {df_clima.provincia.nunique()} provincias × {df_clima.anio.nunique()} años')
print('⚠️  Actualmente usando datos sintéticos. Descomentar Opción A para datos reales de NASA POWER.')

---
<a id="limpieza"></a>

##  8. Calidad y limpieza de datos

Esta sección documenta cada decisión de limpieza con su justificación epidemiológica.

In [ ]:
# ── 8.1 Limpieza del dataset SNVS ─────────────────────────────────────────────
print('=== REPORTE DE CALIDAD — SNVS ===')
print(f'Filas originales         : {len(df_raw):,}')
print(f'Nulos en cantidad_casos  : {df_raw.cantidad_casos.isna().sum()}')
print(f'Departamentos únicos     : {df_raw.departamento.nunique()}')
print(f'Rango de años            : {df_raw.anio.min()} – {df_raw.anio.max()}')
print()

# Decisión 1: excluir registros sin departamento identificable
# Justificación: no pueden asignarse geográficamente para el modelo espacial
mask_desconocido = df_raw['departamento'].str.lower().str.strip().isin(
    ['desconocido', '*sin dato*', 'sin dato', 'nd', '']
)
n_desconocido = mask_desconocido.sum()
print(f'Filas con departamento desconocido: {n_desconocido:,} → EXCLUIDAS')

# Decisión 2: excluir años fuera del rango de interés
mask_anio = (df_raw['anio'] < 2018) | (df_raw['anio'] > 2025)
n_anio = mask_anio.sum()
print(f'Filas con año fuera de 2018–2025 : {n_anio:,} → EXCLUIDAS')

# Decisión 3: corregir semana 53 → 52 (artefacto de calendario ISO)
n_se53 = (df_raw['semana'] == 53).sum()
df_raw.loc[df_raw['semana'] == 53, 'semana'] = 52
print(f'Filas con SE=53 corregidas a 52  : {n_se53:,}')

# Aplicar filtros
df_snvs = df_raw[~mask_desconocido & ~mask_anio].copy()

# Decisión 4: los duplicados del archivo 2025 (primer y segundo semestre)
# pueden solaparse en SE 1–15. Deduplicar por (anio, semana, departamento).
n_antes = len(df_snvs)
df_snvs = df_snvs.groupby(
    ['anio', 'semana', 'departamento', 'provincia'], as_index=False
)['cantidad_casos'].sum()
n_despues = len(df_snvs)
print(f'Filas después de deduplicación   : {n_despues:,} (antes: {n_antes:,})')

# Reporte final
print(f'\n✅ Dataset SNVS limpio           : {len(df_snvs):,} filas | {df_snvs.cantidad_casos.sum():,} casos')
print(f'   Departamentos únicos          : {df_snvs.departamento.nunique()}')
print(f'   Provincias                    : {df_snvs.provincia.nunique()}')

In [ ]:
# ── 8.2 Limpieza del dataset climático ────────────────────────────────────────
print('=== REPORTE DE CALIDAD — CLIMA ===')
print(f'Filas originales   : {len(df_clima):,}')
print(f'Nulos por columna:')
nulos = df_clima.isnull().sum()
print(nulos[nulos > 0].to_string() if (nulos > 0).any() else '  Ninguno')

# Decisión: valores -999 de NASA POWER (missing) → interpolación lineal por provincia
clima_vars_num = df_clima.select_dtypes(include='number').columns.difference(['anio', 'semana'])
for var in clima_vars_num:
    n_miss = df_clima[var].isna().sum()
    if n_miss > 0:
        df_clima[var] = (df_clima.groupby('provincia')[var]
                                  .transform(lambda x: x.interpolate(method='linear', limit_direction='both')))
        print(f'  Interpolados {n_miss} valores en {var}')

print(f'\n✅ Dataset climático limpio: {len(df_clima):,} filas')

In [ ]:
# ── 8.3 Join epidemiológico × climático ───────────────────────────────────────

# Normalizar nombres de provincia para el join
PROV_MAP = {
    'Ciudad De Buenos Aires': 'CABA',
    'Ciudad de Buenos Aires': 'CABA',
    'Caba': 'CABA',
    'Entre Rios': 'Entre Ríos',
    'Cordoba': 'Córdoba',
    'Tucuman': 'Tucumán',
    'Santiago Del Estero': 'Santiago del Estero',
}
df_snvs['provincia_norm'] = (df_snvs['provincia'].str.strip()
                                                  .str.title()
                                                  .replace(PROV_MAP))

df_master = df_snvs.merge(
    df_clima.rename(columns={'provincia': 'provincia_norm'}),
    on=['anio', 'semana', 'provincia_norm'],
    how='left'
)

cobertura_clima = df_master['temp_media'].notna().mean() * 100
print(f'✅ Dataset maestro: {len(df_master):,} filas')
print(f'   Cobertura climática: {cobertura_clima:.1f}% de las filas')
print(f'   Columnas: {list(df_master.columns)}')
df_master.head(3)

In [ ]:
# ── 8.4 Reporte de calidad final ─────────────────────────────────────────────
print('=' * 55)
print('REPORTE DE CALIDAD — DATASET MAESTRO FINAL')
print('=' * 55)
print(f'Filas totales               : {len(df_master):,}')
print(f'Columnas                    : {df_master.shape[1]}')
print(f'Años cubiertos              : {sorted(df_master.anio.unique())}')
print(f'Departamentos únicos        : {df_master.departamento.nunique()}')
print(f'Provincias                  : {df_master.provincia.nunique()}')
print(f'Total casos confirmados     : {df_master.cantidad_casos.sum():,}')
print(f'Semanas epidemiológicas     : {df_master.semana.min()} – {df_master.semana.max()}')
print()
print('Estadísticas de casos por departamento-semana:')
print(df_master['cantidad_casos'].describe().to_string())
print()
print('Nulos por columna:')
nulos_final = df_master.isnull().sum()
print(nulos_final[nulos_final > 0].to_string() if (nulos_final > 0).any() else '  Ninguno')

---
<a id="exportacion"></a>

## 9. Dataset limpio — exportación

Se exportan dos archivos al Google Drive:
1. **`dengue_dataset_limpio.csv`** — dataset maestro completo
2. **`dengue_dataset_departamentos_top.csv`** — subconjunto de departamentos con ≥ 30 observaciones (para modelado)

In [31]:
# Exportar dataset completo
ruta_completo = OUT_DIR / 'dengue_dataset_limpio.csv'
df_master.to_csv(ruta_completo, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_completo}  ({len(df_master):,} filas)')

# Subconjunto: departamentos con suficiente historia
conteo_depto = df_master.groupby('departamento').size()
deptos_suficientes = conteo_depto[conteo_depto >= 30].index
df_modelado = df_master[df_master.departamento.isin(deptos_suficientes)].copy()

ruta_modelado = OUT_DIR / 'dengue_dataset_departamentos_top.csv'
df_modelado.to_csv(ruta_modelado, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_modelado}  ({len(df_modelado):,} filas | {df_modelado.departamento.nunique()} departamentos)')

# Diccionario de variables
diccionario = pd.DataFrame([
    {'variable': 'anio',              'tipo': 'int',   'descripcion': 'Año calendario'},
    {'variable': 'semana',            'tipo': 'int',   'descripcion': 'Semana epidemiológica ISO (1–52)'},
    {'variable': 'departamento',      'tipo': 'str',   'descripcion': 'Nombre del departamento/partido'},
    {'variable': 'provincia',         'tipo': 'str',   'descripcion': 'Nombre de la provincia'},
    {'variable': 'cantidad_casos',    'tipo': 'int',   'descripcion': 'Casos confirmados de dengue (TARGET)'},
    {'variable': 'temp_media',        'tipo': 'float', 'descripcion': 'Temperatura media 2m (°C) — NASA POWER T2M'},
    {'variable': 'temp_max',          'tipo': 'float', 'descripcion': 'Temperatura máxima 2m (°C) — NASA POWER T2M_MAX'},
    {'variable': 'temp_min',          'tipo': 'float', 'descripcion': 'Temperatura mínima 2m (°C) — NASA POWER T2M_MIN'},
    {'variable': 'temp_rango',        'tipo': 'float', 'descripcion': 'Amplitud térmica diaria (°C) — NASA POWER T2M_RANGE'},
    {'variable': 'humedad_media',     'tipo': 'float', 'descripcion': 'Humedad relativa media 2m (%) — NASA POWER RH2M'},
    {'variable': 'humedad_max',       'tipo': 'float', 'descripcion': 'Humedad relativa máxima (%)'},
    {'variable': 'humedad_min',       'tipo': 'float', 'descripcion': 'Humedad relativa mínima (%)'},
    {'variable': 'precipitacion_mm',  'tipo': 'float', 'descripcion': 'Precipitación diaria corregida (mm) — NASA POWER PRECTOTCORR'},
    {'variable': 'precipitacion_acum','tipo': 'float', 'descripcion': 'Precipitación acumulada semanal (mm)'},
    {'variable': 'radiacion_solar',   'tipo': 'float', 'descripcion': 'Radiación solar incidente (W/m²) — NASA POWER ALLSKY_SFC_SW_DWN'},
    {'variable': 'viento_medio',      'tipo': 'float', 'descripcion': 'Velocidad del viento media 2m (m/s) — NASA POWER WS2M'},
    {'variable': 'humedad_suelo',     'tipo': 'float', 'descripcion': 'Humedad del suelo capa raíz — NASA POWER GWETROOT'},
    {'variable': 'punto_rocio',       'tipo': 'float', 'descripcion': 'Temperatura de punto de rocío (°C) — NASA POWER T2MDEW'},
])

ruta_diccionario = OUT_DIR / 'diccionario_variables.csv'
diccionario.to_csv(ruta_diccionario, index=False, encoding='utf-8-sig')
print(f'✅ Guardado: {ruta_diccionario}')

NameError: name 'df_master' is not defined

---
<a id="eda"></a>

## 10. Análisis exploratorio (EDA)

In [ ]:
# 10.1 Serie temporal nacional
serie = df_master.groupby(['anio', 'semana'])['cantidad_casos'].sum().reset_index()
serie['t'] = (serie['anio'] - 2018) * 52 + serie['semana']

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(serie['t'], serie['cantidad_casos'], alpha=0.2, color=PALETTE[0])
ax.plot(serie['t'], serie['cantidad_casos'], color=PALETTE[0], lw=1.2)
ax.axvspan(6 * 52, 7 * 52, alpha=0.07, color='red', label='Brote 2024 (test set)')
xticks = [i * 52 + 1 for i in range(8)]
ax.set_xticks(xticks)
ax.set_xticklabels(range(2018, 2026), fontsize=9)
ax.set_xlabel('Año')
ax.set_ylabel('Casos confirmados')
ax.set_title('Casos de dengue — suma semanal nacional (SNVS 2018–2025)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(df_master.groupby('anio')['cantidad_casos'].sum().rename('casos_totales').to_frame())

In [ ]:
# 10.2 Estacionalidad + top departamentos + distribución
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Estacionalidad (sin 2024 para no distorsionar)
estac = df_master[df_master.anio < 2024].groupby('semana')['cantidad_casos'].mean()
axes[0].bar(estac.index, estac.values, color=PALETTE[0], alpha=0.8, width=0.9)
axes[0].set_title('Estacionalidad media (2018–2023)')
axes[0].set_xlabel('Semana epidemiológica')
axes[0].set_ylabel('Casos promedio')

# Top 12 departamentos
top_d = (df_master[~df_master.departamento.str.lower().isin(['desconocido'])]
               .groupby('departamento')['cantidad_casos']
               .sum().sort_values(ascending=True).tail(12))
top_d.plot(kind='barh', ax=axes[1], color=PALETTE[1], alpha=0.85)
axes[1].set_title('Top 12 departamentos (2018–2025)')
axes[1].set_xlabel('Casos totales')
axes[1].tick_params(axis='y', labelsize=8)

# Distribución log
vals = df_master[df_master.cantidad_casos > 0]['cantidad_casos']
axes[2].hist(np.log1p(vals), bins=50, color=PALETTE[2], alpha=0.8, edgecolor='white')
axes[2].set_title('Distribución de casos (log+1)')
axes[2].set_xlabel('log(casos + 1)')
axes[2].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

In [ ]:
# 10.3 Heatmap año × semana
pivot = (df_master.groupby(['anio', 'semana'])['cantidad_casos']
               .sum().unstack('semana').fillna(0))

fig, ax = plt.subplots(figsize=(15, 4))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0, cbar_kws={'label': 'Casos'})
ax.set_title('Heatmap casos — suma nacional por año y semana epidemiológica')
ax.set_xlabel('Semana epidemiológica')
ax.set_ylabel('Año')
plt.tight_layout()
plt.show()

In [ ]:
# 10.4 Correlación clima × casos por lag (hipótesis H2)
lag_range  = range(0, 9)
clima_corr = ['temp_media', 'humedad_media', 'precipitacion_mm', 'punto_rocio']
clima_corr = [v for v in clima_corr if v in df_master.columns]

df_sorted = df_master.sort_values(['departamento', 'anio', 'semana'])
corr_df   = pd.DataFrame(index=lag_range, columns=clima_corr, dtype=float)

for lag in lag_range:
    for var in clima_corr:
        shifted = df_sorted.groupby('departamento')[var].shift(lag)
        corr_df.loc[lag, var] = shifted.corr(df_sorted['cantidad_casos'])

fig, ax = plt.subplots(figsize=(9, 4))
for i, var in enumerate(clima_corr):
    ax.plot(corr_df.index, corr_df[var].astype(float), marker='o',
            label=var, color=PALETTE[i], lw=1.8)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Lag (semanas)')
ax.set_ylabel('Correlación de Pearson')
ax.set_title('Correlación climática vs casos — por lag temporal\n(prueba de H2: temperatura como predictor dominante)')
ax.legend(fontsize=9)
ax.set_xticks(list(lag_range))
plt.tight_layout()
plt.show()

lag_optimo = corr_df['temp_media'].astype(float).idxmax()
print(f'Lag óptimo para temp_media: {lag_optimo} semanas (correlación = {corr_df.loc[lag_optimo, "temp_media"]:.3f})')

---
<a id="features"></a>

## 11. Feature engineering

Se construyen tres familias de features según la literatura revisada (Mussumeci & Coelho, 2020; Sebastianelli et al., 2024):

1. **Lags epidemiológicos** — casos en t−1 a t−4 semanas
2. **Clima rezagado** — temperatura, humedad, precipitación en t−2, t−4, t−6 semanas
3. **Features cíclicas y contextuales** — sin/cos semana, indicador verano austral

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['departamento', 'anio', 'semana']).copy()
    g  = df.groupby('departamento')

    # Lags epidemiológicos
    for lag in [1, 2, 3, 4]:
        df[f'casos_lag{lag}'] = g['cantidad_casos'].shift(lag)

    # Media móvil 4 SE (baseline)
    df['casos_ma4'] = g['cantidad_casos'].transform(
        lambda x: x.shift(1).rolling(4, min_periods=1).mean()
    )

    # Clima rezagado
    for var in ['temp_media', 'temp_max', 'humedad_media', 'precipitacion_mm']:
        if var not in df.columns:
            continue
        for lag in [2, 4, 6]:
            df[f'{var}_lag{lag}'] = g[var].shift(lag)

    if 'punto_rocio' in df.columns:
        df['punto_rocio_lag2'] = g['punto_rocio'].shift(2)
        df['punto_rocio_lag4'] = g['punto_rocio'].shift(4)

    if 'humedad_suelo' in df.columns:
        df['humedad_suelo_lag2'] = g['humedad_suelo'].shift(2)

    # Temperatura acumulada (GDD proxy)
    if 'temp_media' in df.columns:
        df['gdd_lag2'] = g['temp_media'].transform(
            lambda x: x.shift(2).rolling(3, min_periods=1).mean()
        )

    # Precipitación acumulada 3 SE
    if 'precipitacion_mm' in df.columns:
        df['precip_acum3_lag2'] = g['precipitacion_mm'].transform(
            lambda x: x.shift(2).rolling(3, min_periods=1).sum()
        )

    # Codificación cíclica de semana
    df['semana_sin'] = np.sin(2 * np.pi * df['semana'] / 52)
    df['semana_cos'] = np.cos(2 * np.pi * df['semana'] / 52)

    # Indicador de verano austral (SE 1–15 y SE 45–52)
    df['es_verano'] = df['semana'].apply(lambda w: 1 if (w <= 15 or w >= 45) else 0)

    # ID numérico de departamento
    df['depto_code'] = df['departamento'].astype('category').cat.codes

    return df


df_feat = build_features(df_modelado)

# Lista de features final (excluye columnas no disponibles)
CANDIDATES = [
    'casos_lag1','casos_lag2','casos_lag3','casos_lag4','casos_ma4',
    'temp_media_lag2','temp_media_lag4','temp_media_lag6',
    'temp_max_lag2','temp_max_lag4',
    'humedad_media_lag2','humedad_media_lag4',
    'precipitacion_mm_lag2','precipitacion_mm_lag4',
    'precip_acum3_lag2','gdd_lag2',
    'punto_rocio_lag2','punto_rocio_lag4',
    'humedad_suelo_lag2',
    'semana_sin','semana_cos','es_verano','depto_code',
]
FEATURE_COLS = [f for f in CANDIDATES if f in df_feat.columns]
TARGET = 'cantidad_casos'

df_model = df_feat[FEATURE_COLS + [TARGET, 'anio', 'departamento', 'casos_ma4']].dropna()
print(f'✅ Dataset modelable: {len(df_model):,} filas | {len(FEATURE_COLS)} features')
print(f'   Burn-in eliminado (NaN por lags): {len(df_feat) - len(df_model):,} filas')
print(f'   Features: {FEATURE_COLS}')

---
<a id="modelos"></a>

## 12. Modelo baseline y modelo candidato

- **Baseline:** media móvil de 4 SE (comparador)
- **Modelo v1:** Random Forest (300 árboles)
- **Modelo v2:** XGBoost (500 estimadores)
- **Test set:** brote 2024

In [ ]:
def mape(y_true: np.ndarray, y_pred: np.ndarray, min_casos: int = 5) -> float:
    """MAPE excluyendo observaciones con menos de min_casos (denominador inestable)."""
    mask = y_true >= min_casos
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


train = df_model[df_model['anio'] < 2024]
test  = df_model[df_model['anio'] == 2024]

X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET].values
y_baseline = test['casos_ma4'].values

# ── Baseline ──────────────────────────────────────────────────────────────────
mae_bl  = mean_absolute_error(y_test, y_baseline)
mape_bl = mape(y_test, y_baseline)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=300, max_depth=10,
                            min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_rf    = np.clip(rf.predict(X_test), 0, None)
mae_rf  = mean_absolute_error(y_test, y_rf)
mape_rf = mape(y_test, y_rf)

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    objective='reg:squarederror', random_state=42, n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)
y_xgb    = np.clip(xgb_model.predict(X_test), 0, None)
mae_xgb  = mean_absolute_error(y_test, y_xgb)
mape_xgb = mape(y_test, y_xgb)

# Tabla comparativa
resultados = pd.DataFrame({
    'Modelo':   ['Baseline (MA4)', 'Random Forest', 'XGBoost'],
    'MAE':      [mae_bl,  mae_rf,  mae_xgb],
    'MAPE (%)': [mape_bl, mape_rf, mape_xgb],
    'Objetivo MAPE ≤ 30%': [
        '—',
        '✅' if mape_rf  <= 30 else '❌',
        '✅' if mape_xgb <= 30 else '❌',
    ]
})
print('=== Resultados en test set (brote 2024) ===')
print(resultados.to_string(index=False))

---
<a id="validacion"></a>

## 13. Validación walk-forward

In [ ]:
wf_results = []
eval_years = [y for y in sorted(df_model.anio.unique()) if y >= 2020]

for pred_year in eval_years:
    tr = df_model[df_model['anio'] < pred_year]
    te = df_model[df_model['anio'] == pred_year]
    if len(tr) < 100 or len(te) == 0:
        continue

    # Random Forest
    m_rf = RandomForestRegressor(n_estimators=150, max_depth=8,
                                  min_samples_leaf=5, random_state=42, n_jobs=-1)
    m_rf.fit(tr[FEATURE_COLS], tr[TARGET])
    pred_rf = np.clip(m_rf.predict(te[FEATURE_COLS]), 0, None)

    # XGBoost
    m_xgb = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
                               subsample=0.8, random_state=42, n_jobs=-1, verbosity=0)
    m_xgb.fit(tr[FEATURE_COLS], tr[TARGET])
    pred_xgb = np.clip(m_xgb.predict(te[FEATURE_COLS]), 0, None)

    wf_results.append({
        'año':          pred_year,
        'n_obs':        len(te),
        'mape_baseline':mape(te[TARGET].values, te['casos_ma4'].values),
        'mape_rf':      mape(te[TARGET].values, pred_rf),
        'mape_xgb':     mape(te[TARGET].values, pred_xgb),
        'mae_rf':       mean_absolute_error(te[TARGET], pred_rf),
        'mae_xgb':      mean_absolute_error(te[TARGET], pred_xgb),
    })
    print(f'  {pred_year}: baseline {wf_results[-1]["mape_baseline"]:.1f}% | RF {wf_results[-1]["mape_rf"]:.1f}% | XGB {wf_results[-1]["mape_xgb"]:.1f}%')

df_wf = pd.DataFrame(wf_results)
df_wf['mejora_rf_pp']  = df_wf['mape_baseline'] - df_wf['mape_rf']
df_wf['mejora_xgb_pp'] = df_wf['mape_baseline'] - df_wf['mape_xgb']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(df_wf['año'], df_wf['mape_baseline'], 'o--', color='gray',   label='Baseline (MA4)', lw=1.5)
axes[0].plot(df_wf['año'], df_wf['mape_rf'],       'o-',  color=PALETTE[0], label='Random Forest', lw=2)
axes[0].plot(df_wf['año'], df_wf['mape_xgb'],      's-',  color=PALETTE[1], label='XGBoost',       lw=2)
axes[0].axhline(30, color=PALETTE[2], lw=1, ls=':', alpha=0.8, label='Objetivo ≤ 30%')
axes[0].set_xlabel('Año de predicción')
axes[0].set_ylabel('MAPE (%)')
axes[0].set_title('Validación walk-forward — MAPE por año')
axes[0].legend(fontsize=9)

x = np.arange(len(df_wf))
w = 0.35
axes[1].bar(x - w/2, df_wf['mejora_rf_pp'],  w, label='RF vs baseline',  color=PALETTE[0], alpha=0.85)
axes[1].bar(x + w/2, df_wf['mejora_xgb_pp'], w, label='XGB vs baseline', color=PALETTE[1], alpha=0.85)
axes[1].axhline(0, color='gray', lw=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_wf['año'])
axes[1].set_ylabel('Reducción MAPE (pp)')
axes[1].set_title('Mejora sobre baseline (pp)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
<a id="resultados"></a>

##  14. Resultados e interpretabilidad

In [ ]:
# Importancia de features — Random Forest
imp_rf = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

# Importancia XGBoost
imp_xgb = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp_rf.tail(15).plot(kind='barh', ax=axes[0], color=PALETTE[0], alpha=0.85)
axes[0].set_xlabel('Importancia (Gini)')
axes[0].set_title('Top 15 features — Random Forest')

imp_xgb.tail(15).plot(kind='barh', ax=axes[1], color=PALETTE[1], alpha=0.85)
axes[1].set_xlabel('Importancia (Gain)')
axes[1].set_title('Top 15 features — XGBoost')

plt.tight_layout()
plt.show()

print('Top 5 RF:',  imp_rf.tail(5).sort_values(ascending=False).index.tolist())
print('Top 5 XGB:', imp_xgb.tail(5).sort_values(ascending=False).index.tolist())

In [ ]:
# Predicción vs real — departamento de mayor carga 2024
deptos_2024 = test.groupby('departamento')[TARGET].sum().sort_values(ascending=False)
depto_foco  = deptos_2024.index[0]
print(f'Departamento con mayor carga en 2024: {depto_foco} ({deptos_2024.iloc[0]:.0f} casos)')

mask      = test['departamento'] == depto_foco
sem_d     = test.loc[mask, 'semana'].values
y_real_d  = y_test[mask.values]
y_rf_d    = y_rf[mask.values]
y_xgb_d   = y_xgb[mask.values]
y_base_d  = y_baseline[mask.values]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sem_d, y_real_d, 'o-', color='black',    lw=2,   label='Casos reales',  zorder=4)
ax.plot(sem_d, y_rf_d,   's--', color=PALETTE[0], lw=1.8, label='Random Forest', zorder=3)
ax.plot(sem_d, y_xgb_d,  'd-.', color=PALETTE[1], lw=1.8, label='XGBoost',       zorder=3)
ax.plot(sem_d, y_base_d, '^:',  color='gray',     lw=1.4, label='Baseline MA4',   zorder=2)
ax.fill_between(sem_d, y_xgb_d * 0.7, y_xgb_d * 1.3, alpha=0.1, color=PALETTE[1], label='XGB ±30%')
ax.set_xlabel('Semana epidemiológica')
ax.set_ylabel('Casos confirmados')
ax.set_title(f'{depto_foco} — predicción sobre brote 2024')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Resumen final del estado del proyecto
print('═' * 60)
print('RESUMEN FINAL — ESTADO DEL PROYECTO')
print('═' * 60)
print(f'Dataset SNVS (filas limpias)    : {len(df_master):,}')
print(f'Total casos 2018–2025           : {df_master.cantidad_casos.sum():,}')
print(f'Departamentos modelados         : {df_model.departamento.nunique()}')
print(f'Features construidas            : {len(FEATURE_COLS)}')
print()
print(f'Resultados en test set (2024):')
print(f'  Baseline (MA4)   MAPE = {mape_bl:.1f}%')
print(f'  Random Forest    MAPE = {mape_rf:.1f}%   (mejora: {mape_bl - mape_rf:+.1f} pp)')
print(f'  XGBoost          MAPE = {mape_xgb:.1f}%   (mejora: {mape_bl - mape_xgb:+.1f} pp)')
print()
print('Próximos pasos:')
print('  1. Conectar NASA POWER real (descomentar Opción A sección 7)')
print('  2. Incorporar NDVI via Google Earth Engine (Sentinel-2)')
print('  3. Agregar NBI y densidad del Censo 2022')
print('  4. Implementar corrección de subregistro (~1.5×)')
print('  5. Análisis regional NOA vs pampeana vs NEA (H3)')
print('  6. Evaluación de alerta binaria (F1, precisión) para operaciones')